# 🎙️ SpeakerScribe · **v5 unificado** — transcripción + diarización de nivel producción para Google Colab

Un único notebook que reemplaza a los cuatro anteriores (`batch_recursivo`, `sol_ultra`, `v3.2`, `v4`): el **núcleo transaccional auditado** de v3.2 + la **capa de entregables seleccionables** de v4 + mejoras que ninguno tenía.

| Garantía | Cómo |
|---|---|
| 📂 Recursivo sin límite, `.ts` incluido | `os.walk` + `ffprobe` clasificado (*inválido permanente* vs *transitorio*: un tropiezo de Drive no condena un archivo) |
| 🪞 Espejo con nombres limpios y **estables** | `A/reunion.ts → A/reunion.txt` · homónimos `(ext)` · plantilla `{stem} {carpeta} {modelo} {fecha}` · los nombres ya publicados se conservan |
| 📄 **7 entregables seleccionables** | `.txt` ⭐ · `.transcript.md` · `.srt` · `.json` · `.full_for_llm.txt` · `splits` · `.plano.txt` — activables uno a uno |
| ♻️ **Cambiar de formato sale gratis** | JSON maestro comprimido por archivo: activar un entregable, cambiar plantilla, encabezado o timestamps **republica en CPU** (segundos), sin gastar GPU |
| 🔐 Identidad SHA-256 **completa e incremental** | dedup por contenido (mismo audio en 2 rutas = copia sin GPU) + auditoría rotatoria que caza sustituciones con igual tamaño y fecha |
| 💾 Transaccional de verdad | eventos JSONL + `fsync` · commit `prepared→completed` · recuperación por conjunto-pendiente · lock cerrado con heartbeat y verificación de propiedad |
| 🛡️ **Guardia de RAM** | fuga real medida (~71 MB/archivo): recicla modelos al 70 % y cierra LIMPIO al 88 % **antes** del OOM — nada se pierde |
| 🩺 Caja negra | `telemetria.jsonl` (RAM/VRAM/disco/etapa cada minuto) + celda de **autopsia** que dictamina OOM vs corte externo |
| 🧯 Válvula anti-pérdida | tras agotar reintentos publica el `.txt` **MARCADO** y lo lista en `pendientes_revision.json` — nunca te quedas sin nada, nunca en silencio |
| 🩹 Reparación de hablantes huérfanos | community-1 entrega diarización exclusiva: los tramos sin turno se **adoptan del vecino** antes de juzgar calidad |
| ⏱️ Presupuesto predictivo | RTF aprendido (p10 real): no empieza un audio de 2 h con 20 min restantes |
| 👤 Renombrar hablantes | `SPEAKER_00 → Ana` **persistente**: sobrevive a republicaciones futuras |
| 🔌 Apagado garantizado | en `finally`, cancelable con ⏹; nunca tras `abort_fatal` ni `limite_ram` (necesitas la máquina viva para diagnosticar) |

**Motor por defecto (máxima precisión práctica):** `large-v3` · beam 5 · batch 8 · asignación por palabra · anti-alucinación (`condition_on_previous_text=False`) · VAD 1 s.
Medido en este proyecto: beam 10 duplica el costo (RTF 13.4x vs 21–28x) por una ganancia de WER no demostrada; batch 8 rinde 2–4x con paridad práctica (batch 1 = ruta secuencial bit-exacta).

## ✅ Antes de ejecutar

1. **GPU**: `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU` (u otra GPU: el notebook se adapta).
2. **HF_TOKEN** (tipo *Read*, no fine-grained) en **Secrets** 🔑 con acceso al notebook + términos aceptados en [pyannote/speaker-diarization-community-1](https://huggingface.co/pyannote/speaker-diarization-community-1) · token: <https://huggingface.co/settings/tokens>.
3. Repo, zip o **wheel de SpeakerScribe 0.3.x** en Drive (`RUTA_SPEAKERSCRIBE`, CELDA 2).
4. Edita rutas en la **CELDA 3** → `Ejecutar todo` (Ctrl+F9).
5. *(Opcional)* Glosario por archivo: `<nombre>.prompt.txt` junto al medio — lo más importante **al final** (Whisper solo condiciona con los ~224 tokens finales).

> 🔁 La primera instalación puede pedir **un** reinicio planificado; al volver, `Ejecutar todo` de nuevo (fast-path por huella del código: segundos).
> ⏱️ Cortes de Colab: nada confirmado se pierde; la próxima sesión retoma sola. **Límite honesto:** la unidad transaccional es un archivo — el audio en curso al morir la sesión se reinicia; los anteriores no.
> 🏷️ `SPEAKER_00`, `SPEAKER_01`… distinguen voces dentro de cada archivo; no identifican personas reales (usa la CELDA 10 para asignar nombres).

## 🟢 1 · Montar Google Drive

Va primero: la librería, las salidas y todo el estado de reanudación viven en tu Drive. Los WAV temporales van al disco local de la VM (no pagan la latencia de Drive).

In [ ]:
# 🟢 CELDA 1 — montar Google Drive (estado, salidas y librería viven aquí)
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

## 🟢 2 · Instalar SpeakerScribe 0.3.x (wheel, zip o repo · fast-path por huella del código)

`RUTA_SPEAKERSCRIBE` puede apuntar a: (a) la raíz del repositorio, (b) `speakerscribe-main.zip`, (c) un wheel `speakerscribe-*.whl`, o (d) una carpeta que contenga cualquiera de ellos (se elige el más reciente y se anuncia).

**Por qué puede pedir un reinicio (y por qué está bien):** `pyannote.audio` 4.0.x ancla `torch` a una versión exacta y pip a veces actualiza `numpy` — binarios que la sesión ya tiene cargados. La celda lo detecta y reinicia **una sola vez** (lo instalado persiste). Al volver, `Ejecutar todo`: el fast-path por huella SHA-256 del código termina en segundos y detecta incluso cambios de código con el mismo número de versión.

> ⚠️ No hagas `%cd` a la carpeta de la librería: una copia `speakerscribe/` en el CWD ensombrece al paquete instalado. La celda sanea `sys.path`/CWD y verifica el origen real del import.

In [ ]:
# 🟢 CELDA 2 — instalación (wheel/zip/repo) · fast-path por huella del código · a lo sumo UN reinicio
REINICIO_AUTOMATICO = True   # ⇦ False = solo avisa y tú reinicias manualmente

import hashlib
import importlib
import importlib.metadata as importlib_metadata
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

from packaging.version import Version

RUTA_SPEAKERSCRIBE = '/content/drive/MyDrive/Pruebas/Speakerscribe'  # ⇦ repo, speakerscribe-main.zip o wheel 0.3.x
MIN_V, MAX_V = Version('0.3.0'), Version('0.4.0')
_MARCADOR = Path('/content/.ss_v5_install_marker.json')
_CRITICOS = ('numpy', 'torch', 'torchaudio', 'torchvision', 'pydantic')
os.environ.setdefault('HF_HUB_DISABLE_TELEMETRY', '1')


def _ver(paquete: str):
    """Versión instalada del paquete, o None si no existe."""
    try:
        return Version(importlib_metadata.version(paquete))
    except importlib_metadata.PackageNotFoundError:
        return None


def _dentro(p, base: Path) -> bool:
    try:
        rp = Path(p or '.').resolve()
    except OSError:
        return False
    return rp == base or base in rp.parents


def _extraer_zip_seguro(archivo: Path, destino: Path) -> Path:
    """Extrae validando que ningún miembro escape del destino (zip-slip)."""
    destino.mkdir(parents=True, exist_ok=True)
    destino_res = destino.resolve()
    with zipfile.ZipFile(archivo) as zf:
        for m in zf.infolist():
            objetivo = (destino / m.filename).resolve()
            if objetivo != destino_res and destino_res not in objetivo.parents:
                raise RuntimeError(f'ZIP inseguro: {m.filename!r} sale del destino')
        zf.extractall(destino)
    return destino


def _origen_instalacion(raw: str) -> str:
    """Resuelve wheel/zip/repo. Ante varios candidatos elige el MÁS RECIENTE y lo anuncia."""
    if raw.startswith(('https://', 'git+https://')):
        print('⚠️  Origen por URL: no es reproducible (contenido mutable). '
              'Para producción, descarga el wheel/zip a Drive y apunta ahí.')
        return raw
    root = Path(raw).expanduser()
    if not root.exists():
        raise FileNotFoundError(f'No existe RUTA_SPEAKERSCRIBE: {root}\n'
                                'Apunta al repositorio, a speakerscribe-main.zip o a un wheel 0.3.x.')
    if root.is_file() and root.suffix == '.whl':
        return str(root)
    if root.is_file() and root.suffix.lower() == '.zip':
        destino = Path('/content/ss_v5_install_source')
        if destino.exists():
            shutil.rmtree(destino)
        _extraer_zip_seguro(root, destino)
        pyp = sorted(destino.rglob('pyproject.toml'))
        if not pyp:
            raise FileNotFoundError(f'{root.name} no contiene pyproject.toml')
        if len(pyp) > 1:
            print(f'⚠️  {len(pyp)} proyectos en el zip; se usa {pyp[0].parent.name}.')
        return str(pyp[0].parent)
    if root.is_dir():
        wheels = sorted(root.rglob('speakerscribe-*.whl'), key=lambda p: p.stat().st_mtime_ns)
        if wheels:
            if len(wheels) > 1:
                print(f'⚠️  {len(wheels)} wheels; se usa el más reciente: {wheels[-1].name}. '
                      'Apunta al wheel exacto para fijarlo.')
            return str(wheels[-1])
        if (root / 'pyproject.toml').exists():
            return str(root)
        pyp = sorted(root.rglob('pyproject.toml'))
        if pyp:
            if len(pyp) > 1:
                print(f'⚠️  {len(pyp)} repositorios bajo la ruta; se usa {pyp[0].parent}.')
            return str(pyp[0].parent)
        zips = sorted(root.rglob('speakerscribe*.zip'), key=lambda p: p.stat().st_mtime_ns)
        if zips:
            return _origen_instalacion(str(zips[-1]))
    raise FileNotFoundError(f'Sin wheel, zip o pyproject de SpeakerScribe bajo {root}')


def _huella_origen(objetivo: str) -> str:
    """Identifica el CÓDIGO, no solo el número de versión (detecta cambios silenciosos)."""
    if objetivo.startswith(('https://', 'git+https://')):
        return hashlib.sha256(objetivo.encode()).hexdigest()
    ruta = Path(objetivo).resolve()
    h = hashlib.sha256()
    archivos = [ruta] if ruta.is_file() else (
        [p for p in (ruta / 'pyproject.toml', ruta / 'setup.py') if p.is_file()]
        + sorted((ruta / 'speakerscribe').rglob('*.py')))
    if not archivos:
        raise FileNotFoundError(f'Sin código instalable en {ruta}')
    for f in archivos:
        rel = f.name if ruta.is_file() else f.relative_to(ruta).as_posix()
        h.update(rel.encode() + b'\0')
        with f.open('rb') as s:
            for bloque in iter(lambda: s.read(1 << 20), b''):
                h.update(bloque)
        h.update(b'\0')
    return h.hexdigest()


def _verificar_origen_import() -> str:
    """El import debe resolver al paquete INSTALADO, no a una copia local/Drive."""
    import speakerscribe
    real = Path(speakerscribe.__file__).resolve().parent
    esperado = Path(importlib_metadata.distribution('speakerscribe')
                    .locate_file('speakerscribe')).resolve()
    if real != esperado:
        raise RuntimeError('El import resuelve a una copia local que ensombrece la instalada '
                           f'({real} ≠ {esperado}). Sal del directorio del repo y reinicia.')
    return speakerscribe.__version__


def _pip_check_stack() -> None:
    """pip check filtrado a nuestro stack (la base de Colab trae quejas ajenas)."""
    chk = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True)
    stack = ('speakerscribe', 'whisper', 'ctranslate', 'pyannote', 'torch', 'numpy',
             'numba', 'pydantic', 'lightning', 'torchmetrics')
    avisos = [l for l in chk.stdout.splitlines() if any(s in l.lower() for s in stack)]
    if avisos:
        for l in avisos:
            print('⚠️  pip check:', l)
    else:
        print('✔ pip check: stack coherente (sin conflictos declarados).')


# Higiene: la carpeta de la librería en Drive no debe ensombrecer al paquete instalado.
_base_local = Path(RUTA_SPEAKERSCRIBE).expanduser()
if _base_local.exists() and _base_local.is_dir():
    _b = _base_local.resolve()
    sys.path[:] = [p for p in sys.path if not _dentro(p, _b)]
    if _dentro(os.getcwd(), _b):
        os.chdir('/content')
        print('CWD movido a /content (estaba dentro de la carpeta de la librería).')

t0 = time.time()
objetivo = _origen_instalacion(RUTA_SPEAKERSCRIBE)
huella = _huella_origen(objetivo)
try:
    marcador = json.loads(_MARCADOR.read_text())
except Exception:
    marcador = {}
inst = _ver('speakerscribe')
mismo_codigo = marcador.get('source_sha256') == huella
ss_cargado = any(n == 'speakerscribe' or n.startswith('speakerscribe.') for n in sys.modules)

if inst and MIN_V <= inst < MAX_V and mismo_codigo:
    v = _verificar_origen_import()
    _pip_check_stack()
    print(f'⚡ SpeakerScribe v{v} ya corresponde EXACTAMENTE al código configurado '
          f'({time.time() - t0:.1f} s) — sigue con la CELDA 3.')
else:
    antes = {p: _ver(p) for p in _CRITICOS}
    ya = {p for p in _CRITICOS if p in sys.modules}
    cmd = [sys.executable, '-m', 'pip', 'install', '--upgrade', objetivo]
    if inst and MIN_V <= inst < MAX_V:
        cmd[4:4] = ['--force-reinstall', '--no-deps']   # misma versión, código distinto
    print(f'Instalando desde: {objetivo}  (~2-3 min la primera vez)')
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:   # pip NUNCA debe fallar en silencio
        print(r.stdout[-6000:])
        print(r.stderr[-4000:])
        raise RuntimeError('pip no pudo instalar SpeakerScribe; el detalle está arriba.')
    importlib.invalidate_caches()
    inst = _ver('speakerscribe')
    if inst is None or not (MIN_V <= inst < MAX_V):
        raise RuntimeError(f'Se requiere SpeakerScribe >=0.3.0,<0.4.0; quedó {inst}')
    _MARCADOR.write_text(json.dumps({'source_sha256': huella, 'target': objetivo}))
    despues = {p: _ver(p) for p in _CRITICOS}
    deriva = [f'{p}: {antes[p]} → {despues[p]}' for p in ya
              if antes[p] and despues[p] and antes[p] != despues[p]]
    if deriva or ss_cargado:
        print('Instalación OK; cambiaron binarios ya cargados:',
              deriva + (['speakerscribe'] if ss_cargado else []))
        print('═' * 62)
        print('  REINICIO PLANIFICADO (1 de 1) — todo lo instalado PERSISTE.')
        print('  Cuando el runtime vuelva (~10 s), usa de nuevo')
        print('  «Entorno de ejecución → Ejecutar todo»: esta celda entra')
        print('  por su fast-path en segundos y el flujo continúa completo.')
        print('═' * 62)
        if REINICIO_AUTOMATICO:
            sys.stdout.flush()
            time.sleep(5)              # da tiempo a que el mensaje se muestre
            os.kill(os.getpid(), 9)    # Colab reinicia el kernel solo; paquetes intactos
        raise RuntimeError('Reinicia el runtime (Entorno de ejecución → Reiniciar sesión) '
                           'y vuelve a «Ejecutar todo».')
    v = _verificar_origen_import()
    _pip_check_stack()
    print(f'SpeakerScribe v{v} listo en {time.time() - t0:.0f} s (sin reinicio necesario).')

## 🟢 3 · Configuración — ✏️ **EDITA SOLO ESTA CELDA**

Siete bloques: **rutas**, **motor**, **diarización**, **descubrimiento**, **entregables**, **identidad/calidad** y **sesión/apagado**. Solo las dos rutas del bloque 1 son obligatorias; todo lo demás tiene defaults de producción.

- El motor viene en **máxima precisión práctica** (`large-v3` · beam 5 · batch 8 · asignación por palabra · anti-alucinación). Medido: beam 10 duplica el costo sin ganancia demostrada; `BATCH_SIZE=1` da la ruta secuencial bit-exacta.
- Cambiar **formato/entregables/plantilla** después de procesar NO gasta GPU: la próxima corrida republica desde el JSON maestro en CPU.
- Cambiar **motor** (modelo, idioma, beam, hablantes…) sí cambia el resultado: `POLITICA_CAMBIO_PERFIL` decide (conservar y reportar · reprocesar · detener).
- La celda no importa nada: puedes editarla y ejecutarla antes de que termine la instalación. La validación estricta ocurre en el preflight.

In [ ]:
# 🟢 CELDA 3 — ✏️ CONFIGURACIÓN ÚNICA (edita aquí) · v5.0
import time as _t
SESION_INICIADA_MONOTONIC = _t.monotonic()   # el presupuesto de sesión cuenta desde AQUÍ

# ═══ 1 · RUTAS (lo único obligatorio) ═══════════════════════════════
RUTA_ENTRADA = '/content/drive/MyDrive/Pruebas/Speakerscribe/data'   # @param {type:"string"}
RUTA_SALIDA = '/content/drive/MyDrive/Pruebas/Speakerscribe/entregables'            # @param {type:"string"}

# ═══ 2 · MOTOR — máxima precisión práctica ══════════════════════════
MODELO = 'large-v3'          # @param ["large-v3", "large-v3-turbo", "medium", "small"]
IDIOMA = 'es'                # @param {type:"string"}   ('' = autodetección por archivo)
BEAM_SIZE = 5                # @param {type:"integer"}  (5 = default whisper. Medido AQUÍ: beam 5 ⇒ RTF 21-28x; beam 10 ⇒ 13.4x, sin ganancia de WER demostrada)
BATCH_SIZE = 8               # @param [1, 2, 4, 8] {type:"raw"}  (8 = 2-4x más audio por sesión; 1 = ruta secuencial bit-exacta; auto-baja si hay OOM)
VAD_MIN_SILENCE_MS = 1000    # @param {type:"integer"}  (silencios ≥ N ms cortan segmento)
ANTI_ALUCINACION = True      # @param {type:"boolean"}  (condition_on_previous_text=False: corta bucles de repetición en audio largo/ruidoso)
GLOSARIO_GLOBAL = ''         # @param {type:"string"}  (nombres propios/jerga, lo más importante AL FINAL; por archivo: <nombre>.prompt.txt)

# ═══ 3 · DIARIZACIÓN (0 = autodetección / sin cota) ═════════════════
NUM_HABLANTES = 0            # @param {type:"integer"}  (si conoces el número EXACTO, decláralo: mejora la precisión. No adivines)
MIN_HABLANTES = 1            # @param {type:"integer"}
MAX_HABLANTES = 20           # @param {type:"integer"}
REPARAR_HABLANTES_HUERFANOS = True  # @param {type:"boolean"}  (community-1 deja tramos sin turno: se adoptan del vecino)
TOLERANCIA_HUERFANO_S = 2.0  # @param {type:"number"}   (distancia máxima al vecino para heredar hablante)
MAX_FRACCION_SIN_HABLANTE = 0.35    # @param {type:"number"}  (red de seguridad TRAS la reparación; 0.05 rechazaba material legítimo)
ACEPTAR_AUDIO_SIN_VOZ = True # @param {type:"boolean"}  (audio silencioso ⇒ línea SIN_VOZ, sin gastar reintentos)

# ═══ 4 · DESCUBRIMIENTO RECURSIVO ═══════════════════════════════════
BUSCAR_RECURSIVO = True          # @param {type:"boolean"}  (False = solo la raíz de RUTA_ENTRADA)
EXTENSIONES_SOLO = ''            # @param {type:"string"}  ('ts' ⇒ SOLO .ts; vacío = todas las conocidas)
EXTENSIONES_ADICIONALES = ''     # @param {type:"string"}
EXTENSIONES_EXCLUIR = ''         # @param {type:"string"}
PROBAR_EXTENSIONES_DESCONOCIDAS = True   # @param {type:"boolean"}  (ffprobe decide si un archivo raro tiene audio)
MAX_DESCONOCIDOS_A_PROBAR = 5000         # @param {type:"integer"}
SEGUIR_ENLACES_SIMBOLICOS = False        # @param {type:"boolean"}
SEGUNDOS_ESTABILIDAD_ARCHIVO = 15        # @param {type:"integer"}  (un archivo recién modificado se pospone hasta estabilizarse)

# ═══ 5 · ENTREGABLES — activa lo que necesites ══════════════════════
SALIDA_TXT_TIMESTAMPS = True   # @param {type:"boolean"}  ⭐ [HH:MM:SS - HH:MM:SS] HABLANTE: texto
SALIDA_TRANSCRIPT_MD = False   # @param {type:"boolean"}  Markdown legible por bloques de hablante
SALIDA_SRT = False             # @param {type:"boolean"}  subtítulos
SALIDA_JSON = False            # @param {type:"boolean"}  metadata + segmentos (auditoría)
SALIDA_FULL_LLM = False        # @param {type:"boolean"}  texto corrido con hablantes, sin timestamps (pegar en un LLM)
SALIDA_SPLITS = False          # @param {type:"boolean"}  trozos de N palabras para LLMs de contexto corto
SALIDA_TXT_PLANO = False       # @param {type:"boolean"}  una línea por segmento, sin timestamps
ESPEJO_SUBCARPETAS = True      # @param {type:"boolean"}  (False = todo plano en RUTA_SALIDA)
PLANTILLA_NOMBRE = '{stem}'    # @param {type:"string"}   (marcadores: {stem} {carpeta} {modelo} {fecha})
INCLUIR_ENCABEZADO = True      # @param {type:"boolean"}  (cabecera '#' con origen/duración/idioma/hablantes)
TIMESTAMP_MILISEGUNDOS = False # @param {type:"boolean"}  (False = [HH:MM:SS], menos tokens para LLMs)
MULETILLAS_MD = 'safe'         # @param ["off", "safe", "aggressive"]  (solo filtra el .transcript.md; el .txt conserva TODO)
GAP_BLOQUE_MD_S = 3.0          # @param {type:"number"}   (pausa > N s del mismo hablante abre bloque nuevo en el .md)
SPLIT_PALABRAS = 1950          # @param {type:"integer"}  (tamaño de cada split)
GUARDAR_JSON_MAESTRO = True    # @param {type:"boolean"}  (⭐ habilita republicar/renombrar SIN GPU; ~0.3-1 MB por hora de audio)

# ═══ 6 · IDENTIDAD, REANUDACIÓN Y CALIDAD ═══════════════════════════
MODO_HUELLA = 'full'         # @param ["full", "fast"]  (full: SHA-256 completo incremental = dedup por contenido y detección de sustituciones. fast: muestreado, SIN dedup)
VERIFICAR_HASH_SALIDA = True # @param {type:"boolean"}  (valida el sha256 de cada .txt confirmado antes de darlo por bueno)
AUDITORIAS_ESTRICTAS_POR_SESION = 12     # @param {type:"integer"}  (re-lee N confirmados por sesión: caza sustituciones con igual tamaño+fecha)
MAX_GB_AUDITORIA_POR_SESION = 15.0       # @param {type:"number"}
POLITICA_CAMBIO_PERFIL = 'conservar_y_reportar'  # @param ["conservar_y_reportar", "reprocesar", "detener"]
FORZAR_REPROCESO = False     # @param {type:"boolean"}  (rehace TODO e ignora la caché durable de diarización)
MAX_INTENTOS_POR_ARCHIVO = 2 # @param {type:"integer"}  (solo protege fallos TRANSITORIOS: el pipeline es determinista)
RECHAZAR_CALIDAD_CRITICA = True          # @param {type:"boolean"}
PUBLICAR_DEGRADADO_TRAS_AGOTAR = True    # @param {type:"boolean"}  (🧯 publica MARCADO y lo lista en pendientes_revision.json)
PUBLICAR_SI_CALIDAD_NO_MEJORA = True     # @param {type:"boolean"}  (el rechazo de calidad es determinista: publica marcado en la 1ª pasada en vez de quemar otra sesión)

# ═══ 7 · SESIÓN, RAM, BLOQUEO Y APAGADO ═════════════════════════════
MAX_MINUTOS_SESION = 650     # @param {type:"integer"}  (0 = sin límite; 650 cierra ordenado antes del tope de Colab)
MARGEN_CIERRE_MINUTOS = 20   # @param {type:"integer"}
MAX_ARCHIVOS_POR_SESION = 0  # @param {type:"integer"}  (0 = sin límite; 2 = corrida de prueba)
TAMANO_BLOQUE_PLANIFICACION = 1  # @param {type:"integer"}  (1 = el primer TXT sale lo antes posible)
RTF_ASR_CONSERVADOR = 8.0    # @param {type:"number"}  (solo mientras no haya ≥5 métricas propias; después se usa el RTF p10 real)
RTF_DIAR_CONSERVADOR = 10.0  # @param {type:"number"}
SOBRECOSTE_ARCHIVO_MIN = 5   # @param {type:"integer"}
RAM_PCT_RECICLAR = 70        # @param {type:"integer"}  (fuga medida ~71 MB/archivo: al superarlo, libera y recarga modelos ~35 s)
RAM_PCT_DETENER = 88         # @param {type:"integer"}  (cierre LIMPIO antes del OOM; nada se pierde)
RECICLAR_CADA_N_ARCHIVOS = 0 # @param {type:"integer"}  (0 = solo por RAM)
VIGILANCIA_RECURSOS = True   # @param {type:"boolean"}  (caja negra: RAM/VRAM/disco a telemetria.jsonl cada minuto)
LATIDO_MINUTOS = 10          # @param {type:"integer"}  (señal de vida periódica; 0 = sin latido)
APAGAR_AL_FINAL = True       # @param {type:"boolean"}
APAGAR_TRAS_ERROR_FATAL = False  # @param {type:"boolean"}  (⚠️ True mata la VM ante CUALQUIER excepción y destruye la evidencia)
APAGAR_SOLO_SI_HUBO_TRABAJO = True  # @param {type:"boolean"}  (terminar en <10 min sin entregables es un síntoma, no un éxito)
RETRASO_APAGADO_S = 180      # @param {type:"integer"}  (cuenta regresiva cancelable con ⏹)
FORZAR_TOMAR_BLOQUEO = False # @param {type:"boolean"}
DEPURAR_GUARDAR_LOGS_FALLO = True  # @param {type:"boolean"}

# ── Validación temprana (falla aquí, en segundos — no a mitad del lote) ──
assert MODELO in ('large-v3', 'large-v3-turbo', 'medium', 'small'), f'MODELO inválido: {MODELO}'
assert int(BEAM_SIZE) >= 1 and int(BATCH_SIZE) in (1, 2, 4, 8)
assert int(VAD_MIN_SILENCE_MS) >= 100
assert MODO_HUELLA in ('fast', 'full')
assert POLITICA_CAMBIO_PERFIL in ('conservar_y_reportar', 'reprocesar', 'detener')
assert MULETILLAS_MD in ('off', 'safe', 'aggressive')
assert 0.0 <= float(MAX_FRACCION_SIN_HABLANTE) <= 1.0
assert float(TOLERANCIA_HUERFANO_S) >= 0 and float(GAP_BLOQUE_MD_S) >= 0
assert int(SPLIT_PALABRAS) >= 100
assert int(MAX_INTENTOS_POR_ARCHIVO) >= 1 and int(RETRASO_APAGADO_S) >= 0
assert int(TAMANO_BLOQUE_PLANIFICACION) >= 1 and int(SOBRECOSTE_ARCHIVO_MIN) >= 0
assert float(RTF_ASR_CONSERVADOR) > 0 and float(RTF_DIAR_CONSERVADOR) > 0
assert MAX_MINUTOS_SESION == 0 or 0 <= MARGEN_CIERRE_MINUTOS < MAX_MINUTOS_SESION
assert int(AUDITORIAS_ESTRICTAS_POR_SESION) >= 0 and float(MAX_GB_AUDITORIA_POR_SESION) > 0
assert 0 < int(RAM_PCT_RECICLAR) < int(RAM_PCT_DETENER) <= 95, \
    'RAM_PCT_RECICLAR debe ser menor que RAM_PCT_DETENER, y este ≤ 95'
assert int(RECICLAR_CADA_N_ARCHIVOS) >= 0 and int(LATIDO_MINUTOS) >= 0
assert any((SALIDA_TXT_TIMESTAMPS, SALIDA_TRANSCRIPT_MD, SALIDA_SRT, SALIDA_JSON,
            SALIDA_FULL_LLM, SALIDA_SPLITS, SALIDA_TXT_PLANO)), \
    'Activa al menos un entregable (SALIDA_* en el bloque 5).'
_marc = set(__import__('re').findall(r'{(\w+)}', PLANTILLA_NOMBRE))
assert _marc and 'stem' in _marc and _marc <= {'stem', 'carpeta', 'modelo', 'fecha'}, \
    "PLANTILLA_NOMBRE debe incluir {stem}; marcadores válidos: {stem} {carpeta} {modelo} {fecha}"
if not GUARDAR_JSON_MAESTRO:
    print('⚠️  GUARDAR_JSON_MAESTRO=False: republicar formatos y renombrar hablantes '
          'exigirá REPROCESAR EN GPU. No recomendado.')
if MODO_HUELLA == 'fast':
    print('⚠️  MODO_HUELLA=fast: identidad muestreada ⇒ dedup por contenido DESACTIVADA '
          'y sustituciones exóticas indetectables. Úsalo solo a sabiendas.')

print('📂', RUTA_ENTRADA, '→', RUTA_SALIDA)
print('🧠', MODELO, '| beam', BEAM_SIZE, '| batch', BATCH_SIZE,
      '(bit-exacto)' if int(BATCH_SIZE) == 1 else '(2-4x, paridad práctica)',
      '| VAD', VAD_MIN_SILENCE_MS, 'ms | idioma', IDIOMA or 'auto',
      '| anti-alucinación', ANTI_ALUCINACION)
print('👥 hablantes', NUM_HABLANTES or f'{MIN_HABLANTES}-{MAX_HABLANTES}',
      '| reparación huérfanos', REPARAR_HABLANTES_HUERFANOS,
      f'(±{TOLERANCIA_HUERFANO_S}s) | tope sin hablante {float(MAX_FRACCION_SIN_HABLANTE):.0%}')
_activos = [n for n, v in (('txt', SALIDA_TXT_TIMESTAMPS), ('md', SALIDA_TRANSCRIPT_MD),
                           ('srt', SALIDA_SRT), ('json', SALIDA_JSON),
                           ('full_llm', SALIDA_FULL_LLM), ('splits', SALIDA_SPLITS),
                           ('plano', SALIDA_TXT_PLANO)) if v]
print('📄 entregables:', ', '.join(_activos), f'| plantilla "{PLANTILLA_NOMBRE}"',
      '| espejo', ESPEJO_SUBCARPETAS, '| ms', TIMESTAMP_MILISEGUNDOS,
      '| json maestro', GUARDAR_JSON_MAESTRO)
print('🔐 huella', MODO_HUELLA, '| auditorías/sesión', AUDITORIAS_ESTRICTAS_POR_SESION,
      f'(≤{MAX_GB_AUDITORIA_POR_SESION} GB) | política perfil:', POLITICA_CAMBIO_PERFIL)
print(f'🛡️ RAM: recicla >{RAM_PCT_RECICLAR}% · detiene >{RAM_PCT_DETENER}%',
      '| 🩺 vigilancia', VIGILANCIA_RECURSOS,
      '| sesión', f'{MAX_MINUTOS_SESION} min (margen {MARGEN_CIERRE_MINUTOS})'
      if MAX_MINUTOS_SESION else 'sin límite',
      '| apagado', APAGAR_AL_FINAL)

## 🟢 4 · Infraestructura transaccional (solo definiciones — no edites)

Identidad de archivos, eventos JSONL con `fsync`, lock cerrado con heartbeat, telemetría, guardia de RAM, descubrimiento recursivo, nombres estables y commits atómicos. **No carga modelos.** Todo lo configurable vive en la CELDA 3.

In [ ]:
# ===== 🟢 CELDA 4 — INFRAESTRUCTURA TRANSACCIONAL (solo definiciones) =====
"""SpeakerScribe v5 — capa transaccional del lote recursivo.

Contexto: Google Colab (GPU, ~12 GB RAM, sesión ≤12 h) con Drive montado.
Responsabilidad: identidad por contenido, eventos JSONL durables, lock de
sesión, telemetría, descubrimiento y commits atómicos. No carga modelos.

Contrato de durabilidad: la unidad transaccional es UN archivo. Si Colab
corta la sesión se pierde a lo sumo el audio en curso; lo confirmado nunca.

Autor: Enrique Forero · Notebook v5.0.0 · motor: speakerscribe >=0.3,<0.4
"""
from __future__ import annotations

import bisect
import contextlib
import gc
import gzip
import hashlib
import importlib.util
import json
import os
import re
import shutil
import socket
import subprocess
import threading
import time
import traceback
import unicodedata
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

IN_COLAB = importlib.util.find_spec('google.colab') is not None
NOTEBOOK_VERSION = '5.0.0'
OUTPUT_SCHEMA = 'ss-v5-render-1'   # versión del RENDER: al cambiarla, todo se republica en CPU
MANIFEST_SCHEMA = 5
RUN_ID = uuid.uuid4().hex

# ── Rutas de estado (autocontenidas dentro de RUTA_SALIDA) ───────────
INPUT_ROOT = Path(RUTA_ENTRADA).expanduser().resolve()
OUTPUT_ROOT = Path(RUTA_SALIDA).expanduser().resolve()
STATE_ROOT = OUTPUT_ROOT / '.speakerscribe_state'
EVENTS_PATH = STATE_ROOT / 'events.jsonl'
LOCK_PATH = STATE_ROOT / 'active.lock.json'
IDENTITY_PATH = STATE_ROOT / 'workspace_identity.json'
AUDIT_CURSOR_PATH = STATE_ROOT / 'audit_cursor.json'
REVIEW_PATH = STATE_ROOT / 'pendientes_revision.json'
GLOBAL_DIAR_CACHE = STATE_ROOT / 'diar_cache'
INTERMEDIOS_ROOT = STATE_ROOT / 'intermedios'    # JSON maestro .json.gz por fuente
RENOMBRES_ROOT = STATE_ROOT / 'renombres'        # mapping persistente de hablantes por fuente
FAIL_LOGS_ROOT = STATE_ROOT / 'logs_fallos'
SUMMARY_PATH = STATE_ROOT / 'last_summary.json'
TELEMETRY_PATH = STATE_ROOT / 'telemetria.jsonl'
RESUMEN_MD_PATH = OUTPUT_ROOT / '_resumen.md'
SCRATCH_ROOT = (Path('/content/ss_v5_scratch') if IN_COLAB else Path('/tmp/ss_v5_scratch')) \
    / hashlib.sha256(str(OUTPUT_ROOT).encode()).hexdigest()[:12]
JOBS_ROOT = SCRATCH_ROOT / 'jobs'
ALIAS_ROOT = SCRATCH_ROOT / 'aliases'
ACTIVE_LOCK: 'StateLock | None' = None
ARCHIVO_EN_CURSO: str | None = None
ETAPA_EN_CURSO: str | None = None

KNOWN_MEDIA_EXTENSIONS = frozenset({
    '.3gp', '.3g2', '.aac', '.ac3', '.aif', '.aiff', '.amr', '.ape',
    '.asf', '.avi', '.caf', '.dts', '.eac3', '.f4a', '.f4v', '.flac',
    '.m2ts', '.m4a', '.m4v', '.mka', '.mkv', '.mov', '.mp2', '.mp3',
    '.mp4', '.mpeg', '.mpg', '.mts', '.oga', '.ogg', '.ogv', '.opus',
    '.rm', '.rmvb', '.ts', '.vob', '.wav', '.webm', '.wma', '.wmv',
})
OBVIOUS_NON_MEDIA = frozenset({
    '.txt', '.md', '.json', '.jsonl', '.csv', '.tsv', '.xlsx', '.xls',
    '.doc', '.docx', '.pdf', '.ppt', '.pptx', '.py', '.ipynb', '.html',
    '.htm', '.xml', '.yaml', '.yml', '.ini', '.cfg', '.log', '.zip',
    '.rar', '.7z', '.tar', '.gz', '.bz2', '.xz', '.jpg', '.jpeg', '.png',
    '.gif', '.bmp', '.webp', '.svg', '.exe', '.dll', '.so', '.whl',
    '.srt', '.vtt',
})


# ── Utilidades base ──────────────────────────────────────────────────
def nfc(v: str) -> str:
    """Normaliza a NFC (Drive/macOS mezclan NFC y NFD en nombres con tildes)."""
    return unicodedata.normalize('NFC', v)


def _parse_ext_list(raw: str) -> frozenset[str]:
    out = set()
    for v in re.split(r'[,;\s]+', (raw or '').strip()):
        if v:
            out.add(v.lower() if v.startswith('.') else '.' + v.lower())
    return frozenset(out)


def resolved_extensions() -> tuple[frozenset[str], bool, frozenset[str]]:
    """(aceptadas, sondear_desconocidas, excluidas) según la CELDA 3."""
    solo = _parse_ext_list(EXTENSIONES_SOLO)
    excluidas = _parse_ext_list(EXTENSIONES_EXCLUIR)
    if solo:
        return solo, False, excluidas
    aceptadas = (KNOWN_MEDIA_EXTENSIONS | _parse_ext_list(EXTENSIONES_ADICIONALES)) - excluidas
    return frozenset(aceptadas), bool(PROBAR_EXTENSIONES_DESCONOCIDAS), excluidas


@dataclass(frozen=True)
class SourceInfo:
    """Identidad y stat de un archivo fuente + su glosario sidecar."""
    path: Path
    relative: Path
    source_id: str
    size_bytes: int
    mtime_ns: int
    prompt_path: Path | None
    prompt_size: int | None
    prompt_mtime_ns: int | None


@dataclass(frozen=True)
class Job:
    """Unidad de trabajo: fuente + firma de contenido + perfiles + destino."""
    source: SourceInfo
    signature: str
    profile_id: str
    profile: dict[str, Any]
    job_id: str
    output_relative: Path
    duration_seconds: float | None


class SourceChangingError(RuntimeError):
    """La fuente cambió durante la inspección o el proceso: se pospone."""


class FatalDiarizationSetup(RuntimeError):
    """La diarización no puede inicializar (token/términos/CUDA): corta el lote."""


class DiarizationFileError(RuntimeError):
    """Diarización fallida SOLO para este archivo (reintentable / válvula)."""


class QualityRejectedError(RuntimeError):
    """El resultado no supera las validaciones de calidad."""


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def canonical_hash(value: Any) -> str:
    """SHA-256 de la serialización JSON canónica (claves ordenadas)."""
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     separators=(',', ':')).encode()).hexdigest()


def hash_file(path: Path, mode: str = 'full', chunk_size: int = 8 << 20) -> str:
    """Huella de contenido. 'full' = SHA-256 completo; 'fast' = tamaño + extremos.

    El modo 'fast' se marca con prefijo 'fast:' para que NUNCA se confunda con
    una identidad fuerte (la dedup por contenido exige identidad fuerte).
    """
    if mode not in {'full', 'fast'}:
        raise ValueError(f'MODO_HUELLA inválido: {mode}')
    h = hashlib.sha256()
    if mode == 'fast':
        size = path.stat().st_size
        h.update(str(size).encode())
        muestra = 8 << 20
        with path.open('rb') as s:
            h.update(s.read(muestra))
            if size > 2 * muestra:
                s.seek(size - muestra)
                h.update(s.read(muestra))
            elif size > muestra:
                s.seek(muestra)
                h.update(s.read())
        return 'fast:' + h.hexdigest()
    with path.open('rb') as s:
        for bloque in iter(lambda: s.read(chunk_size), b''):
            h.update(bloque)
    return h.hexdigest()


def fsync_directory(path: Path) -> None:
    """fsync del directorio (durabilidad del rename); tolerante a FUSE."""
    try:
        fd = os.open(path, os.O_RDONLY)
    except OSError:
        return
    try:
        os.fsync(fd)
    except OSError:
        pass
    finally:
        os.close(fd)


def atomic_write_text(path: Path, text: str) -> None:
    """Escritura atómica: tmp + fsync + os.replace. Nunca queda un archivo a medias."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f'.tmp.{RUN_ID}')
    try:
        with tmp.open('w', encoding='utf-8', newline='\n') as s:
            s.write(text)
            s.flush()
            with contextlib.suppress(OSError):
                os.fsync(s.fileno())
        os.replace(tmp, path)
        fsync_directory(path.parent)
    finally:
        with contextlib.suppress(OSError):
            if tmp.exists():
                tmp.unlink()


def atomic_write_json(path: Path, value: Any) -> None:
    atomic_write_text(path, json.dumps(value, ensure_ascii=False, indent=2, default=str))


def atomic_write_gz_json(path: Path, value: Any) -> tuple[int, str]:
    """Escribe JSON comprimido (gzip) de forma atómica. → (tamaño, sha256)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f'.tmp.{RUN_ID}')
    try:
        with gzip.open(tmp, 'wt', encoding='utf-8', compresslevel=6) as s:
            json.dump(value, s, ensure_ascii=False, default=str)
        os.replace(tmp, path)
        fsync_directory(path.parent)
    finally:
        with contextlib.suppress(OSError):
            if tmp.exists():
                tmp.unlink()
    return path.stat().st_size, hash_file(path, 'full')


def read_gz_json(path: Path) -> dict[str, Any] | None:
    """Lee un .json.gz (o .json plano como respaldo). None si no existe/ilegible."""
    try:
        if path.suffix == '.gz' or path.name.endswith('.json.gz'):
            with gzip.open(path, 'rt', encoding='utf-8') as s:
                return json.load(s)
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return None


def append_event(event: str, **fields: Any) -> dict[str, Any]:
    """Añade un evento al journal (append-only, fsync): la fuente de la verdad."""
    STATE_ROOT.mkdir(parents=True, exist_ok=True)
    record = {'schema_version': MANIFEST_SCHEMA, 'event_id': uuid.uuid4().hex,
              'run_id': RUN_ID, 'timestamp_utc': utc_now(), 'event': event, **fields}
    linea = json.dumps(record, ensure_ascii=False, separators=(',', ':'), default=str)
    with EVENTS_PATH.open('a', encoding='utf-8') as s:
        s.write(linea + '\n')
        s.flush()
        with contextlib.suppress(OSError):
            os.fsync(s.fileno())
    return record


def read_events() -> list[dict[str, Any]]:
    """Lee el journal tolerando la última línea truncada por un corte."""
    if not EVENTS_PATH.exists():
        return []
    out: list[dict[str, Any]] = []
    with EVENTS_PATH.open(encoding='utf-8', errors='replace') as s:
        for n, linea in enumerate(s, 1):
            linea = linea.strip()
            if not linea:
                continue
            try:
                v = json.loads(linea)
            except json.JSONDecodeError:
                print(f'Advertencia: línea incompleta del journal ignorada ({n}).')
                continue
            if isinstance(v, dict):
                out.append(v)
    return out


def sanitize_error(exc: BaseException) -> str:
    """Mensaje de error sin token de Hugging Face y acotado."""
    msg = f'{type(exc).__name__}: {exc}'
    return re.sub(r'hf_[A-Za-z0-9_-]{8,}', 'hf_***redactado***', msg)[:4000]


def is_within(path: Path, parent: Path) -> bool:
    try:
        path.resolve().relative_to(parent.resolve())
        return True
    except (ValueError, OSError):
        return False


def safe_output_path(relative: Path) -> Path:
    """Resuelve una ruta relativa DENTRO de la salida; aborta cualquier escape."""
    if relative.is_absolute() or '..' in relative.parts:
        raise RuntimeError(f'Ruta de salida insegura: {relative}')
    candidato = (OUTPUT_ROOT / relative).resolve()
    if not is_within(candidato, OUTPUT_ROOT):
        raise RuntimeError(f'La salida resolvería fuera de RUTA_SALIDA: {relative}')
    return candidato


def bind_workspace_identity() -> dict[str, Any]:
    """Liga la carpeta de salida a UNA raíz de entrada: impide pisar resultados
    de otra colección por un error de configuración."""
    identidad = {'schema_version': 1, 'input_root': str(INPUT_ROOT),
                 'input_root_sha256': canonical_hash(str(INPUT_ROOT))}
    if IDENTITY_PATH.exists():
        try:
            previa = json.loads(IDENTITY_PATH.read_text(encoding='utf-8'))
        except Exception as exc:
            raise RuntimeError(f'No se pudo validar {IDENTITY_PATH}; no se tocará la salida.') from exc
        if previa.get('input_root_sha256') != identidad['input_root_sha256']:
            raise RuntimeError(
                'RUTA_SALIDA ya está vinculada a OTRA entrada. Usa otra carpeta de salida.\n'
                f'  Registrada: {previa.get("input_root")}\n  Actual:     {INPUT_ROOT}')
        return previa
    raices_previas = {(r.get('settings') or {}).get('input_root_sha256')
                      for r in read_events() if r.get('event') == 'batch_started'}
    raices_previas.discard(None)
    if raices_previas and identidad['input_root_sha256'] not in raices_previas:
        raise RuntimeError('El historial de esta salida corresponde a otra raíz de entrada.')
    atomic_write_json(IDENTITY_PATH, identidad)
    return identidad


def source_payload(info: SourceInfo, signature: str | None = None) -> dict[str, Any]:
    return {'id': info.source_id, 'relative_path': info.relative.as_posix(),
            'size_bytes': info.size_bytes, 'mtime_ns': info.mtime_ns,
            'content_signature': signature, 'signature_mode': MODO_HUELLA,
            'prompt_path': (info.prompt_path.relative_to(INPUT_ROOT).as_posix()
                            if info.prompt_path is not None else None),
            'prompt_size': info.prompt_size, 'prompt_mtime_ns': info.prompt_mtime_ns}


def source_stat_matches(record: dict[str, Any], info: SourceInfo) -> bool:
    src = record.get('source') or {}
    return (src.get('size_bytes') == info.size_bytes
            and src.get('mtime_ns') == info.mtime_ns
            and src.get('prompt_size') == info.prompt_size
            and src.get('prompt_mtime_ns') == info.prompt_mtime_ns)


def output_path_from_record(record: dict[str, Any]) -> Path | None:
    rel = (record.get('output') or {}).get('relative_path')
    if not rel:
        return None
    try:
        candidato = (OUTPUT_ROOT / rel).resolve()
    except OSError:
        return None
    return candidato if is_within(candidato, OUTPUT_ROOT) else None


def output_record_is_valid(record: dict[str, Any]) -> bool:
    """El .txt principal del registro existe y (si se exige) su sha256 coincide."""
    try:
        path = output_path_from_record(record)
        meta = record.get('output') or {}
        if path is None or not path.is_file() or path.stat().st_size <= 0:
            return False
        if meta.get('size_bytes') is not None and path.stat().st_size != meta['size_bytes']:
            return False
        if VERIFICAR_HASH_SALIDA:
            if not meta.get('sha256'):
                return False
            return hash_file(path, 'full') == meta['sha256']
        return True
    except OSError:
        return False           # tropiezo transitorio de Drive: afecta a esta fuente, no al lote


def output_record_stat_is_valid(record: dict[str, Any]) -> bool:
    """Chequeo barato (sin hash) para ordenar/priorizar; no declara validez final."""
    try:
        path = output_path_from_record(record)
        esperado = (record.get('output') or {}).get('size_bytes')
        return bool(path and path.is_file() and path.stat().st_size > 0
                    and (esperado is None or path.stat().st_size == esperado))
    except OSError:
        return False


class StateLock:
    """Lock cerrado: un archivo ilegible NO se roba salvo FORZAR_TOMAR_BLOQUEO.
    Heartbeat tolerante a tropiezos transitorios; propiedad verificable antes
    de cada promoción de resultado."""

    def __init__(self, path: Path, stale_after_s: int = 600):
        self.path = path
        self.stale_after_s = stale_after_s
        self.owner = RUN_ID
        self.stop_event = threading.Event()
        self.thread: threading.Thread | None = None
        self.lost_reason: str | None = None

    def _payload(self) -> dict[str, Any]:
        return {'owner': self.owner, 'pid': os.getpid(), 'host': socket.gethostname(),
                'heartbeat_epoch': time.time(), 'heartbeat_utc': utc_now()}

    def acquire(self) -> None:
        self.path.parent.mkdir(parents=True, exist_ok=True)
        for intento in range(2):
            try:
                datos = json.dumps(self._payload(), ensure_ascii=False).encode()
                fd = os.open(self.path, os.O_WRONLY | os.O_CREAT | os.O_EXCL, 0o600)
                try:
                    os.write(fd, datos)
                    with contextlib.suppress(OSError):
                        os.fsync(fd)
                finally:
                    os.close(fd)
                break
            except FileExistsError:
                try:
                    previo = json.loads(self.path.read_text(encoding='utf-8'))
                    edad = time.time() - float(previo.get('heartbeat_epoch', 0))
                except Exception as exc:
                    if not FORZAR_TOMAR_BLOQUEO:
                        raise RuntimeError(
                            'El lock existente es ilegible; por seguridad NO se toma '
                            'automáticamente. Verifica que no haya otra sesión y usa '
                            f'FORZAR_TOMAR_BLOQUEO=True. Detalle: {type(exc).__name__}: {exc}'
                        ) from exc
                    previo, edad = {}, float('inf')
                if not FORZAR_TOMAR_BLOQUEO and edad <= self.stale_after_s:
                    raise RuntimeError(
                        'Ya hay otra ejecución activa sobre esta salida '
                        f'(owner={str(previo.get("owner", "?"))[:8]}, heartbeat hace {edad:.0f} s). '
                        'Si es un lock huérfano: FORZAR_TOMAR_BLOQUEO=True.')
                if intento == 0:
                    os.replace(self.path, self.path.with_name(
                        self.path.name + f'.stale.{int(time.time())}'))
                    continue
                raise
        self.thread = threading.Thread(target=self._heartbeat, daemon=True)
        self.thread.start()

    def _heartbeat(self) -> None:
        while not self.stop_event.wait(60):
            try:
                actual = json.loads(self.path.read_text(encoding='utf-8'))
                if actual.get('owner') != self.owner:
                    self.lost_reason = 'el lock pasó a otro propietario'
                    return
                atomic_write_json(self.path, self._payload())
            except Exception:
                continue          # tropiezo transitorio de Drive: reintenta al minuto

    def assert_owned(self) -> None:
        if self.lost_reason:
            raise RuntimeError(f'Se perdió el lock de estado: {self.lost_reason}')
        for intento in (1, 2):
            try:
                actual = json.loads(self.path.read_text(encoding='utf-8'))
                if actual.get('owner') != self.owner:
                    self.lost_reason = 'el propietario del lock no coincide'
                    raise RuntimeError(f'Se perdió el lock de estado: {self.lost_reason}')
                return
            except RuntimeError:
                raise
            except Exception:
                if intento == 2:
                    raise RuntimeError('Se perdió el lock de estado: ilegible dos veces seguidas')
                time.sleep(2)

    def release(self) -> None:
        self.stop_event.set()
        if self.thread is not None:
            self.thread.join(timeout=2)
        try:
            actual = json.loads(self.path.read_text(encoding='utf-8'))
            if actual.get('owner') == self.owner:
                self.path.unlink()
        except Exception:
            pass


# ══ Caja negra de recursos — sin esto, la muerte de una sesión es adivinanza ══
def ram_pct_actual() -> float:
    """Porcentaje de RAM del sistema en uso (0.0 si no se puede medir).

    Lee /proc/meminfo directamente: sin dependencias y sin costo en el camino
    caliente. Una sonda rota nunca debe detener el lote que vigila.
    """
    try:
        info: dict[str, float] = {}
        for linea in Path('/proc/meminfo').read_text().splitlines():
            k, _, v = linea.partition(':')
            info[k] = float(v.split()[0])
        total, disponible = info['MemTotal'], info['MemAvailable']
        return round(100.0 * (total - disponible) / total, 1)
    except Exception:
        return 0.0


def liberar_memoria_host() -> None:
    """gc + vaciar caché CUDA + `malloc_trim(0)` para devolver heap al SO.

    Sin malloc_trim, la RAM liberada por Python queda retenida en las arenas
    de glibc y el porcentaje del sistema no baja aunque el objeto ya no exista.
    """
    gc.collect()
    with contextlib.suppress(Exception):
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    with contextlib.suppress(Exception):
        import ctypes
        ctypes.CDLL('libc.so.6').malloc_trim(0)


def muestra_recursos() -> dict[str, Any]:
    """Fotografía JSON-serializable de RAM/VRAM/disco y etapa en curso. Nunca lanza."""
    inicio = float(globals().get('SESION_INICIADA_MONOTONIC', time.monotonic()))
    d: dict[str, Any] = {'utc': utc_now(), 'run_id': RUN_ID,
                         'min_sesion': round((time.monotonic() - inicio) / 60, 2)}
    with contextlib.suppress(Exception):
        info: dict[str, float] = {}
        for linea in Path('/proc/meminfo').read_text().splitlines():
            k, _, v = linea.partition(':')
            info[k] = float(v.split()[0]) * 1024
        d['ram_total_gb'] = round(info['MemTotal'] / 1e9, 2)
        d['ram_usada_gb'] = round((info['MemTotal'] - info['MemAvailable']) / 1e9, 2)
        d['ram_pct'] = round(100 * d['ram_usada_gb'] / d['ram_total_gb'], 1)
    with contextlib.suppress(Exception):
        import torch
        if torch.cuda.is_available():
            d['vram_asignada_gb'] = round(torch.cuda.memory_allocated() / 1e9, 2)
            d['vram_reservada_gb'] = round(torch.cuda.memory_reserved() / 1e9, 2)
    with contextlib.suppress(Exception):
        base = SCRATCH_ROOT if SCRATCH_ROOT.exists() else Path('/')
        d['disco_libre_gb'] = round(shutil.disk_usage(base).free / 1e9, 2)
    d['archivo'] = globals().get('ARCHIVO_EN_CURSO')
    d['etapa'] = globals().get('ETAPA_EN_CURSO')
    return d


class VigilanteRecursos(threading.Thread):
    """Registra recursos en `telemetria.jsonl` cada `intervalo_s` segundos.

    Cuando una VM de Colab desaparece, el notebook no deja rastro de la causa.
    Si la ÚLTIMA línea muestra la RAM subiendo hacia el tope, fue OOM. Si
    muestra todo plano, el corte fue externo (límite de Colab, desconexión).
    Sin ese dato, cualquier diagnóstico es adivinanza.
    """

    def __init__(self, ruta: Path, intervalo_s: int = 60, latido_min: int = 10) -> None:
        super().__init__(daemon=True)
        self.ruta = ruta
        self.intervalo_s = max(15, int(intervalo_s))
        self.latido_min = max(0, int(latido_min))
        self._parar = threading.Event()
        self.pico_ram_gb = 0.0
        self.ultima: dict[str, Any] = {}

    def _escribir(self, motivo: str) -> None:
        d = muestra_recursos()
        d['motivo'] = motivo
        self.ultima = d
        self.pico_ram_gb = max(self.pico_ram_gb, float(d.get('ram_usada_gb') or 0))
        with contextlib.suppress(Exception):
            self.ruta.parent.mkdir(parents=True, exist_ok=True)
            with self.ruta.open('a', encoding='utf-8') as f:
                f.write(json.dumps(d, ensure_ascii=False) + '\n')
                f.flush()
                with contextlib.suppress(OSError):
                    os.fsync(f.fileno())

    def _emitir(self, texto: str) -> None:
        try:
            from tqdm.auto import tqdm as _tq
            _tq.write(texto)
        except Exception:
            print(texto, flush=True)

    def run(self) -> None:
        self._escribir('inicio')
        siguiente_latido = self.latido_min
        while not self._parar.wait(self.intervalo_s):
            self._escribir('periodico')
            d = self.ultima
            if self.latido_min and d.get('min_sesion', 0) >= siguiente_latido:
                siguiente_latido += self.latido_min
                self._emitir(
                    f"   🩺 {d.get('min_sesion', 0):.0f} min · RAM "
                    f"{d.get('ram_usada_gb', '?')}/{d.get('ram_total_gb', '?')} GB "
                    f"({d.get('ram_pct', '?')}%) · VRAM {d.get('vram_reservada_gb', '?')} GB "
                    f"· disco {d.get('disco_libre_gb', '?')} GB libres · {d.get('etapa') or '—'}")

    def detener(self, motivo: str = 'fin') -> None:
        self._parar.set()
        self._escribir(motivo)


def marcar_etapa(archivo: str | None, etapa: str | None) -> None:
    """Publica qué se está haciendo, para que la telemetría lo grabe."""
    global ARCHIVO_EN_CURSO, ETAPA_EN_CURSO
    ARCHIVO_EN_CURSO, ETAPA_EN_CURSO = archivo, etapa


# ── Descubrimiento recursivo ─────────────────────────────────────────
def prompt_info_for(path: Path) -> tuple[Path | None, int | None, int | None]:
    prompt = path.with_suffix('.prompt.txt')
    try:
        if not prompt.is_file():
            return None, None, None
        st = prompt.stat()
    except OSError:               # sidecar imposible (nombre al límite de 255 bytes)
        return None, None, None
    return prompt, st.st_size, st.st_mtime_ns


def discover_sources() -> tuple[list[SourceInfo], list[str], set[tuple[str, str]]]:
    """Recorre RUTA_ENTRADA (a cualquier profundidad si BUSCAR_RECURSIVO).

    Returns:
        (fuentes ordenadas, avisos, claves de colisión de stem por carpeta).
        Las colisiones se detectan sobre nombres normalizados NFC.
    """
    if not INPUT_ROOT.is_dir():
        raise FileNotFoundError(f'La carpeta de entrada no existe: {INPUT_ROOT}')
    aceptadas, sondear, excluidas = resolved_extensions()
    avisos: list[str] = []
    encontrados: list[SourceInfo] = []
    desconocidos, aviso_tope = 0, False
    root_id = canonical_hash(str(INPUT_ROOT))
    salida_anidada = OUTPUT_ROOT != INPUT_ROOT and is_within(OUTPUT_ROOT, INPUT_ROOT)
    visitados: set[Path] = set()

    def onerror(exc: OSError) -> None:
        avisos.append(f'No se pudo recorrer {getattr(exc, "filename", "?")}: {exc}')

    for dirpath, dirnames, filenames in os.walk(
            INPUT_ROOT, topdown=True, onerror=onerror,
            followlinks=SEGUIR_ENLACES_SIMBOLICOS):
        actual = Path(dirpath)
        actual_real = actual.resolve()
        if not is_within(actual_real, INPUT_ROOT):
            avisos.append(f'Se omite directorio enlazado fuera de la entrada: {actual}')
            dirnames[:] = []
            continue
        if actual_real in visitados:
            dirnames[:] = []
            continue
        visitados.add(actual_real)
        if not BUSCAR_RECURSIVO:
            dirnames[:] = []
        else:
            conservar = []
            for d in dirnames:
                hijo = actual / d
                if d.startswith('.') or hijo == STATE_ROOT or is_within(hijo, STATE_ROOT):
                    continue
                if salida_anidada and (hijo == OUTPUT_ROOT or is_within(hijo, OUTPUT_ROOT)):
                    continue
                if hijo.is_symlink() and not SEGUIR_ENLACES_SIMBOLICOS:
                    continue
                conservar.append(d)
            dirnames[:] = conservar

        for nombre in filenames:
            if nombre.startswith(('.', '~')) or nombre.endswith('.prompt.txt'):
                continue
            path = actual / nombre
            if path.is_symlink() and not SEGUIR_ENLACES_SIMBOLICOS:
                continue
            if path.is_symlink() and not is_within(path.resolve(), INPUT_ROOT):
                avisos.append(f'Se omite enlace que sale de la entrada: {path}')
                continue
            if not path.is_file():
                continue
            suf = path.suffix.lower()
            if suf not in aceptadas:
                if suf in excluidas or not sondear or suf in OBVIOUS_NON_MEDIA:
                    continue
                desconocidos += 1
                if desconocidos > MAX_DESCONOCIDOS_A_PROBAR:
                    if not aviso_tope:
                        avisos.append('Tope MAX_DESCONOCIDOS_A_PROBAR alcanzado: se omiten '
                                      'más extensiones desconocidas; el escaneo de formatos '
                                      'conocidos continúa.')
                        aviso_tope = True
                    continue
            try:
                st = path.stat()
                p_path, p_size, p_mtime = prompt_info_for(path)
                relativo = path.relative_to(INPUT_ROOT)
            except OSError as exc:
                avisos.append(f'No se pudo inspeccionar {path}: {exc}')
                continue
            if st.st_size == 0:
                continue
            encontrados.append(SourceInfo(
                path=path, relative=relativo,
                source_id=canonical_hash({'root': root_id, 'relative': relativo.as_posix()}),
                size_bytes=st.st_size, mtime_ns=st.st_mtime_ns,
                prompt_path=p_path, prompt_size=p_size, prompt_mtime_ns=p_mtime))
    encontrados.sort(key=lambda i: (nfc(i.relative.as_posix()).casefold(), i.source_id))

    vistos: dict[tuple[str, str], int] = {}
    for info in encontrados:
        clave = (nfc(info.relative.parent.as_posix()).casefold(),
                 nfc(info.relative.stem).casefold())
        vistos[clave] = vistos.get(clave, 0) + 1
    colisiones = {k for k, n in vistos.items() if n > 1}
    return encontrados, avisos, colisiones


_FFPROBE_PERMANENTE = ('invalid data found when processing input', 'moov atom not found',
                       'could not find codec parameters', 'end of file')


def ffprobe_audio(path: Path) -> tuple[str, float | None, str | None]:
    """→ ('ok'|'invalid'|'retryable', duración_s, detalle).

    Un tropiezo transitorio de Drive NO condena al archivo como inválido
    permanente: solo los marcadores de corrupción conocidos marcan 'invalid'.
    """
    cmd = ['ffprobe', '-v', 'error', '-select_streams', 'a:0',
           '-show_entries', 'stream=codec_name:format=duration', '-of', 'json', str(path)]
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=60)
    except subprocess.TimeoutExpired:
        return 'retryable', None, 'ffprobe excedió 60 s'
    except OSError as exc:
        return 'retryable', None, f'ffprobe no ejecutable: {exc}'
    if r.returncode != 0:
        detalle = r.stderr.strip()[:1000]
        estado = 'invalid' if any(m in detalle.lower() for m in _FFPROBE_PERMANENTE) else 'retryable'
        return estado, None, detalle
    try:
        payload = json.loads(r.stdout)
        streams = payload.get('streams') or []
        dur_raw = (payload.get('format') or {}).get('duration')
        dur = float(dur_raw) if dur_raw not in (None, 'N/A') else None
    except Exception as exc:
        return 'retryable', None, f'respuesta ffprobe inválida: {exc}'
    if not streams:
        return 'invalid', dur, 'el contenedor no tiene stream de audio'
    return 'ok', dur, None


def stable_signature(info: SourceInfo) -> tuple[str, SourceInfo]:
    """Huella de contenido tomada sobre un archivo ESTABLE (stat antes/después)."""
    antes = info.path.stat()
    edad = time.time() - antes.st_mtime
    if edad < SEGUNDOS_ESTABILIDAD_ARCHIVO:
        raise SourceChangingError(f'modificado hace {edad:.1f} s; se pospone hasta estabilizar')
    firma = hash_file(info.path, MODO_HUELLA)
    despues = info.path.stat()
    if (antes.st_size, antes.st_mtime_ns) != (despues.st_size, despues.st_mtime_ns):
        raise SourceChangingError('el archivo cambió mientras se calculaba su huella')
    p_path, p_size, p_mtime = prompt_info_for(info.path)
    return firma, SourceInfo(path=info.path, relative=info.relative, source_id=info.source_id,
                             size_bytes=despues.st_size, mtime_ns=despues.st_mtime_ns,
                             prompt_path=p_path, prompt_size=p_size, prompt_mtime_ns=p_mtime)


def prompt_hash(info: SourceInfo) -> str:
    """Huella del glosario efectivo (sidecar del archivo o global)."""
    if info.prompt_path is not None:
        return hash_file(info.prompt_path, 'full')
    return hashlib.sha256(GLOSARIO_GLOBAL.strip().encode()).hexdigest()


def source_still_unchanged(info: SourceInfo) -> bool:
    try:
        st = info.path.stat()
        p_path, p_size, p_mtime = prompt_info_for(info.path)
    except OSError:
        return False
    return (st.st_size == info.size_bytes and st.st_mtime_ns == info.mtime_ns
            and p_size == info.prompt_size and p_mtime == info.prompt_mtime_ns
            and p_path == info.prompt_path)


# ── Nombres de salida: plantilla + espejo + estabilidad ──────────────
_INVALIDOS_FS = re.compile(r'[\\/:*?"<>|\x00-\x1f]')


def render_nombre(stem: str, rel: Path) -> str:
    """Aplica PLANTILLA_NOMBRE y sanea el resultado para el sistema de archivos."""
    crudo = PLANTILLA_NOMBRE.format(
        stem=stem,
        carpeta=(rel.parent.name or 'raiz'),
        modelo=MODELO,
        fecha=datetime.now().strftime('%Y-%m-%d'),
    )
    return _INVALIDOS_FS.sub('_', crudo).strip().strip('.') or stem


def output_relative_for(info: SourceInfo, colisiones: set[tuple[str, str]]) -> Path:
    """Destino en espejo con plantilla: 'A/reunion.ts' → 'A/reunion.txt'.

    Homónimos de distinta extensión en la MISMA carpeta → 'reunion (ts).txt'.
    Guardas de longitud: nombre ≤240 bytes; ruta completa ≤3000 bytes.
    """
    stem = info.relative.stem
    clave = (nfc(info.relative.parent.as_posix()).casefold(), nfc(stem).casefold())
    if clave in colisiones and info.relative.suffix:
        stem = f'{stem} ({info.relative.suffix.lstrip(".").lower()})'
    nombre = render_nombre(stem, info.relative) + '.txt'
    if len(nombre.encode('utf-8')) > 240:
        margen = 240 - len('.txt') - 14
        nombre = (nombre[:-4].encode('utf-8')[:max(1, margen)].decode('utf-8', errors='ignore')
                  + f'~{info.source_id[:12]}.txt')
    carpeta = info.relative.parent if ESPEJO_SUBCARPETAS else Path('.')
    relativo = carpeta / nombre
    if len(str(OUTPUT_ROOT / relativo).encode('utf-8')) > 3000:
        relativo = Path('_rutas_largas') / info.source_id[:2] / info.source_id / nombre
    return relativo


def format_ts(seconds: float) -> str:
    """Timestamp según TIMESTAMP_MILISEGUNDOS: '01:02:05' o '01:02:05.240'."""
    if TIMESTAMP_MILISEGUNDOS:
        total_ms = max(0, int(round(float(seconds) * 1000)))
        h, resto = divmod(total_ms, 3_600_000)
        m, resto = divmod(resto, 60_000)
        s, ms = divmod(resto, 1000)
        return f'{h:02d}:{m:02d}:{s:02d}.{ms:03d}'
    total = max(0, int(round(float(seconds))))
    h, resto = divmod(total, 3600)
    m, s = divmod(resto, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'


# ── Commits atómicos y recuperación ──────────────────────────────────
def write_complete_part(final_path: Path, text: str, job_id: str) -> tuple[Path, int, str]:
    """Escribe el contenido COMPLETO como `.part` junto al destino final."""
    final_path.parent.mkdir(parents=True, exist_ok=True)
    part = final_path.with_name(final_path.name + f'.part.{job_id[:16]}')
    atomic_write_text(part, text)
    return part, part.stat().st_size, hash_file(part, 'full')


def copy_complete_part(source: Path, final_path: Path, job_id: str,
                       expected_sha256: str | None = None) -> tuple[Path, int, str]:
    """Copia validada como `.part` (para reutilizaciones sin GPU)."""
    final_path.parent.mkdir(parents=True, exist_ok=True)
    part = final_path.with_name(final_path.name + f'.part.{job_id[:16]}')
    tmp = part.with_name(part.name + f'.tmp.{RUN_ID}')
    try:
        with source.open('rb') as src, tmp.open('wb') as dst:
            shutil.copyfileobj(src, dst, length=4 << 20)
            dst.flush()
            with contextlib.suppress(OSError):
                os.fsync(dst.fileno())
        os.replace(tmp, part)
        fsync_directory(part.parent)
    finally:
        with contextlib.suppress(OSError):
            if tmp.exists():
                tmp.unlink()
    digest = hash_file(part, 'full')
    if expected_sha256 is not None and digest != expected_sha256:
        with contextlib.suppress(OSError):
            part.unlink()
        raise RuntimeError('La salida reutilizable cambió durante la copia')
    return part, part.stat().st_size, digest


def promote_part(part: Path, final_path: Path) -> None:
    """`os.replace` atómico del .part al final, verificando la propiedad del lock."""
    if part.parent.resolve() != final_path.parent.resolve():
        raise RuntimeError('Parcial y final deben compartir filesystem')
    if ACTIVE_LOCK is not None:
        ACTIVE_LOCK.assert_owned()
    os.replace(part, final_path)
    fsync_directory(final_path.parent)


def build_indices(events: list[dict[str, Any]]) -> dict[str, Any]:
    """Índices en memoria del journal (se construye UNA vez por sesión)."""
    latest_by_job: dict[str, dict[str, Any]] = {}
    ok_por_fuente: dict[str, list[dict[str, Any]]] = {}
    ok_por_contenido_perfil: dict[tuple[str, str], list[dict[str, Any]]] = {}
    invalidos: dict[str, dict[str, Any]] = {}
    sucias: set[str] = set()
    intentos: dict[str, int] = {}
    renombres_hash: dict[str, str] = {}
    for rec in events:
        job_id = rec.get('job_id')
        if job_id:
            latest_by_job[job_id] = rec
        evento = rec.get('event')
        fuente = rec.get('source') or {}
        sid = fuente.get('id')
        if evento in {'completed', 'reused', 'republished'} and sid:
            sucias.discard(sid)
            ok_por_fuente.setdefault(sid, []).append(rec)
            firma = fuente.get('content_signature')
            perfil = (rec.get('profile') or {}).get('id')
            fuerte = firma and not str(firma).startswith('fast:')
            if fuerte and perfil:   # republished también dona: su sha es el vigente
                ok_por_contenido_perfil.setdefault((firma, perfil), []).append(rec)
        elif evento == 'invalid_media' and sid:
            invalidos[sid] = rec
        elif evento == 'content_change_detected' and sid:
            sucias.add(sid)
        elif evento == 'content_change_resolved' and sid:
            sucias.discard(sid)
        elif evento == 'speakers_renamed' and sid:
            renombres_hash[sid] = str(rec.get('rename_sha') or '')
        # probe_retryable y source_changed NO consumen intentos
        if job_id and evento in {'failed_retryable', 'quality_rejected'}:
            intentos[job_id] = intentos.get(job_id, 0) + 1
    return {'latest_by_job': latest_by_job, 'ok_por_fuente': ok_por_fuente,
            'ok_por_contenido_perfil': ok_por_contenido_perfil,
            'invalidos': invalidos, 'sucias': sucias, 'intentos': intentos,
            'renombres_hash': renombres_hash}


def recover_prepared(events: list[dict[str, Any]]) -> int:
    """Semántica de conjunto-pendiente: un 'prepared' solo se cierra con
    completed/reused/republished/prepared_abandoned — un fallo posterior no
    entierra un commit rescatable."""
    pendientes: dict[str, dict[str, Any]] = {}
    for rec in events:
        job_id = rec.get('job_id')
        if not job_id:
            continue
        ev = rec.get('event')
        if ev == 'prepared':
            pendientes[job_id] = rec
        elif ev in {'completed', 'reused', 'republished', 'prepared_abandoned'}:
            pendientes.pop(job_id, None)
    recuperados = 0
    for rec in pendientes.values():
        salida = rec.get('output') or {}
        final_rel, part_rel = salida.get('relative_path'), salida.get('part_relative_path')
        if not final_rel:
            continue
        final = (OUTPUT_ROOT / final_rel).resolve()
        part = (OUTPUT_ROOT / part_rel).resolve() if part_rel else None
        if not is_within(final, OUTPUT_ROOT) or (part and not is_within(part, OUTPUT_ROOT)):
            continue

        def coincide(p: Path | None) -> bool:
            try:
                return bool(p and p.is_file() and p.stat().st_size == salida.get('size_bytes')
                            and hash_file(p, 'full') == salida.get('sha256'))
            except OSError:
                return False

        if coincide(final):
            if part is not None and part.exists() and part != final:
                with contextlib.suppress(OSError):
                    part.unlink()
        elif coincide(part):
            promote_part(part, final)  # type: ignore[arg-type]
        else:
            append_event('prepared_abandoned', job_id=rec.get('job_id'),
                         source=rec.get('source'), profile=rec.get('profile'),
                         error='No se pudo recuperar el commit preparado')
            continue
        append_event('completed', job_id=rec.get('job_id'), source=rec.get('source'),
                     profile=rec.get('profile'), presentacion=rec.get('presentacion'),
                     extras=rec.get('extras'), master_json=rec.get('master_json'),
                     output={k: v for k, v in salida.items() if k != 'part_relative_path'},
                     quality=rec.get('quality', 'ok'), recovered_after_interruption=True)
        recuperados += 1
    return recuperados


print('🧱 Infraestructura v5 cargada: identidad, journal, lock, telemetría, '
      'descubrimiento, nombres y commits atómicos.')

## 🟢 5 · Render y entregables (solo definiciones — no edites)

Reparación de hablantes huérfanos, renombres persistentes, los 7 renderizadores, el **JSON maestro** comprimido (lo que hace gratis republicar y renombrar) y el perfil de **presentación** — separado del perfil de **motor** para que un cambio de formato nunca queme GPU.

In [ ]:
# ===== 🟢 CELDA 5 — RENDER Y ENTREGABLES (solo definiciones) =====
_HABLANTES_NULOS = {None, '', '(no diarization)', 'SPEAKER_NO_OVERLAP', 'None'}
RENOMBRES_CACHE: dict[str, dict[str, str]] = {}
_RENOMBRES_CARGADOS = False


def reparar_hablantes(segments: list[dict[str, Any]],
                      tolerancia_s: float) -> tuple[list[dict[str, Any]], dict[str, int]]:
    """Adopta los segmentos huérfanos (sin turno de diarización que los cubra).

    Por qué hace falta: `pyannote/speaker-diarization-community-1` devuelve
    diarización EXCLUSIVA — los tramos solapados o de baja confianza quedan
    fuera de todo turno y se etiquetan `SPEAKER_NO_OVERLAP`. Descartar el
    archivo por eso tira a la basura una pasada completa de GPU sin mejorar
    nada, porque el reproceso es determinista y devolvería lo mismo.

    Reglas, de la más segura a la más laxa:
        1. Si la diarización halló UN solo hablante, todo huérfano es suyo.
        2. Si halló varios, el huérfano hereda cuando sus dos vecinos
           temporales coinciden en hablante (sin ambigüedad posible).
        3. Si no coinciden, hereda del vecino más cercano solo cuando la
           distancia es ≤ `tolerancia_s`.
        4. El resto sigue como SIN_HABLANTE y cuenta contra
           MAX_FRACCION_SIN_HABLANTE.

    Args:
        segments: Segmentos con claves `start`, `end`, `text`, `speaker`.
        tolerancia_s: Distancia máxima, en segundos, para heredar de un vecino.

    Returns:
        Tupla `(segmentos, stats)` con `huerfanos`, `adoptados` y
        `hablantes_reales`. Los segmentos de entrada NO se mutan.
    """
    stats = {'huerfanos': 0, 'adoptados': 0, 'hablantes_reales': 0}
    if not segments:
        return segments, stats

    etiquetas: list[str | None] = [
        None if s.get('speaker') in _HABLANTES_NULOS else str(s.get('speaker'))
        for s in segments]
    reales = sorted({e for e in etiquetas if e})
    stats['hablantes_reales'] = len(reales)
    stats['huerfanos'] = sum(1 for e in etiquetas if e is None)
    if not reales or stats['huerfanos'] == 0:
        return segments, stats

    nuevas = list(etiquetas)
    if len(reales) == 1:
        unico = reales[0]
        nuevas = [unico if e is None else e for e in etiquetas]
    else:
        idx_reales = [i for i, e in enumerate(etiquetas) if e]
        for i, e in enumerate(etiquetas):
            if e is not None:
                continue
            pos = bisect.bisect_left(idx_reales, i)
            ant = idx_reales[pos - 1] if pos > 0 else None
            sig = idx_reales[pos] if pos < len(idx_reales) else None
            if ant is not None and sig is not None and etiquetas[ant] == etiquetas[sig]:
                nuevas[i] = etiquetas[ant]
                continue
            candidato, distancia = None, float('inf')
            if ant is not None:
                d = abs(float(segments[i].get('start', 0.0))
                        - float(segments[ant].get('end', 0.0)))
                if d < distancia:
                    candidato, distancia = etiquetas[ant], d
            if sig is not None:
                d = abs(float(segments[sig].get('start', 0.0))
                        - float(segments[i].get('end', 0.0)))
                if d < distancia:
                    candidato, distancia = etiquetas[sig], d
            if candidato is not None and distancia <= float(tolerancia_s):
                nuevas[i] = candidato

    stats['adoptados'] = sum(1 for a, b in zip(etiquetas, nuevas) if a is None and b is not None)
    if not stats['adoptados']:
        return segments, stats
    salida = []
    for seg, etiqueta in zip(segments, nuevas):
        if etiqueta and seg.get('speaker') in _HABLANTES_NULOS:
            seg = dict(seg)
            seg['speaker'] = etiqueta
        salida.append(seg)
    return salida, stats


# ── Renombres persistentes de hablantes (SPEAKER_00 → Ana) ───────────
def _cargar_cache_renombres() -> None:
    """Carga TODOS los mappings una vez por sesión (un solo readdir a Drive)."""
    global _RENOMBRES_CARGADOS
    if _RENOMBRES_CARGADOS:
        return
    _RENOMBRES_CARGADOS = True
    if not RENOMBRES_ROOT.is_dir():
        return
    for f in RENOMBRES_ROOT.glob('*.json'):
        try:
            datos = json.loads(f.read_text(encoding='utf-8'))
            mapping = datos.get('mapping') or {}
            if isinstance(mapping, dict) and mapping:
                RENOMBRES_CACHE[f.stem] = {str(k): str(v) for k, v in mapping.items()}
        except Exception:
            print(f'⚠️  Mapping de renombres ilegible (se ignora): {f.name}')


def cargar_renombres(source_id: str) -> dict[str, str]:
    _cargar_cache_renombres()
    return RENOMBRES_CACHE.get(source_id, {})


def guardar_renombres(source_id: str, mapping: dict[str, str]) -> str:
    """Persiste el mapping y devuelve su huella (entra en el perfil de presentación)."""
    _cargar_cache_renombres()
    mapping = {str(k): str(v) for k, v in mapping.items() if str(k) != str(v)}
    if mapping:
        RENOMBRES_CACHE[source_id] = mapping
        atomic_write_json(RENOMBRES_ROOT / f'{source_id}.json',
                          {'mapping': mapping, 'actualizado': utc_now()})
    else:
        RENOMBRES_CACHE.pop(source_id, None)
        with contextlib.suppress(OSError):
            (RENOMBRES_ROOT / f'{source_id}.json').unlink()
    return rename_sha_de(mapping)


def rename_sha_de(mapping: dict[str, str]) -> str:
    return canonical_hash(mapping) if mapping else ''


def aplicar_renombres(segments: list[dict[str, Any]],
                      mapping: dict[str, str]) -> list[dict[str, Any]]:
    """Aplica el mapping en UNA sola pasada (un swap 00→01, 01→Ana no encadena)."""
    if not mapping:
        return segments
    salida = []
    for seg in segments:
        bruto = seg.get('speaker')
        if bruto is not None and str(bruto) in mapping:
            seg = dict(seg)
            seg['speaker'] = mapping[str(bruto)]
        salida.append(seg)
    return salida


def preparar_segmentos(metadata: dict[str, Any],
                       source_id: str) -> tuple[list[dict[str, Any]], dict[str, int], str]:
    """Segmentos publicables: reparación de huérfanos + renombres persistentes.

    No muta `metadata` — el JSON maestro conserva los segmentos CRUDOS del
    motor (trazabilidad) y esta preparación es determinista y re-aplicable.

    Returns:
        (segmentos, stats_reparacion, rename_sha)
    """
    segs = metadata.get('segments') or []
    stats = {'huerfanos': 0, 'adoptados': 0, 'hablantes_reales': 0}
    if REPARAR_HABLANTES_HUERFANOS and segs:
        segs, stats = reparar_hablantes(segs, float(TOLERANCIA_HUERFANO_S))
    mapping = cargar_renombres(source_id)
    if mapping:
        segs = aplicar_renombres(segs, mapping)
    return segs, stats, rename_sha_de(mapping)


# ── Renderizadores ───────────────────────────────────────────────────
def _encabezado(metadata: dict[str, Any], source_rel: str, estado: str,
                motivo: str | None, hablantes: set[str],
                repair: dict[str, int] | None) -> list[str]:
    prob = metadata.get('language_probability')
    lineas = [
        f'# archivo_origen: {source_rel}',
        f'# duracion: {format_ts(metadata.get("duration_seconds") or 0)}'
        f' · idioma: {metadata.get("language_detected") or metadata.get("language") or "?"}'
        + (f' (p={prob:.2f})' if isinstance(prob, (int, float)) else ''),
        f'# hablantes: {len(hablantes)}'
        + (f' ({", ".join(sorted(hablantes))})' if hablantes else ''),
        f'# modelo: faster-whisper {metadata.get("model", MODELO)} + '
        + str(metadata.get('diarization_model') or 'pyannote community-1')
        + f' · beam {int(BEAM_SIZE)} · batch {int(BATCH_SIZE)}',
        f'# palabras: {metadata.get("total_words", "?")} · estado: {estado}'
        + (f' · motivo: {motivo}' if motivo else ''),
    ]
    if repair and repair.get('adoptados'):
        lineas.append(f'# reparacion_hablantes: {repair["adoptados"]} de '
                      f'{repair["huerfanos"]} segmentos huérfanos adoptados '
                      f'({repair["hablantes_reales"]} hablantes)')
    lineas += [f'# esquema: {OUTPUT_SCHEMA} · notebook v{NOTEBOOK_VERSION} '
               f'· procesado: {utc_now()}',
               '# ' + '─' * 68, '']
    return lineas


def render_transcript(segments: list[dict[str, Any]], metadata: dict[str, Any],
                      source_rel: str, estado: str, motivo: str | None = None,
                      repair: dict[str, int] | None = None) -> str:
    """El `.txt` canónico: `[ini - fin] HABLANTE: texto`, una línea por segmento.

    estado='ok' activa validaciones estrictas (diarización presente,
    monotonicidad, tope de palabras sin hablante). Audio silencioso ⇒ línea
    `SIN_VOZ` de primera clase. Estados degradados: tolerantes pero MARCADOS
    en el encabezado.

    Raises:
        ValueError: solo en modo estricto, cuando el resultado no es publicable.
    """
    estricto = estado == 'ok'
    if estricto and not segments:
        if not ACEPTAR_AUDIO_SIN_VOZ:
            raise ValueError('La transcripción no contiene segmentos')
        dur = max(0.0, float(metadata.get('duration_seconds') or 0.0))
        cuerpo = [f'[{format_ts(0)} - {format_ts(dur)}] SIN_VOZ: (sin voz detectada)']
        cab = _encabezado(metadata, source_rel, 'ok_sin_voz', None, set(), repair) \
            if INCLUIR_ENCABEZADO else []
        return '\n'.join(cab + cuerpo) + '\n'

    if estricto and not metadata.get('diarization_enabled'):
        raise ValueError('La diarización no está presente')
    reales = {str(s.get('speaker')) for s in segments
              if s.get('speaker') not in _HABLANTES_NULOS}
    if estricto and segments and not reales:
        raise ValueError('No se detectó ningún hablante válido')

    lineas: list[str] = []
    previo = -1.0
    palabras_total = palabras_sin = 0
    for idx, seg in enumerate(segments, 1):
        texto = ' '.join(str(seg.get('text') or '').split())
        if not texto:
            continue
        ini = float(seg.get('start', 0.0))
        fin = float(seg.get('end', ini))
        if estricto:
            if fin < ini:
                raise ValueError(f'Segmento {idx} termina antes de empezar')
            if ini + 0.05 < previo:
                raise ValueError(f'Timestamps no monotónicos en segmento {idx}')
        else:
            fin = max(fin, ini)
        previo = max(previo, ini)
        bruto = seg.get('speaker')
        hablante = 'SIN_HABLANTE' if bruto in _HABLANTES_NULOS else str(bruto)
        n = len(texto.split())
        palabras_total += n
        if hablante == 'SIN_HABLANTE':
            palabras_sin += n
        lineas.append(f'[{format_ts(ini)} - {format_ts(fin)}] {hablante}: {texto}')
    if estricto and palabras_total:
        frac = palabras_sin / palabras_total
        if frac > float(MAX_FRACCION_SIN_HABLANTE):
            raise ValueError(f'{frac:.1%} de palabras sin hablante '
                             f'(máx {float(MAX_FRACCION_SIN_HABLANTE):.1%})')
    if not lineas:
        if estricto:
            raise ValueError('No quedó texto publicable')
        lineas.append('(sin voz detectada)')
    cab = _encabezado(metadata, source_rel, estado, motivo, reales, repair) \
        if INCLUIR_ENCABEZADO else []
    return '\n'.join(cab + lineas) + '\n'


def _ts_srt(seconds: float) -> str:
    total_ms = max(0, int(round(float(seconds) * 1000)))
    h, resto = divmod(total_ms, 3_600_000)
    m, resto = divmod(resto, 60_000)
    s, ms = divmod(resto, 1000)
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'


def render_srt(segments: list[dict[str, Any]]) -> str:
    """Subtítulos SRT con numeración consecutiva y hablante como prefijo."""
    bloques: list[str] = []
    n = 0
    for seg in segments:
        texto = ' '.join(str(seg.get('text') or '').split())
        if not texto:
            continue
        n += 1
        ini = float(seg.get('start', 0.0))
        fin = max(float(seg.get('end', ini)), ini)
        bruto = seg.get('speaker')
        prefijo = '' if bruto in _HABLANTES_NULOS else f'{bruto}: '
        bloques.append(f'{n}\n{_ts_srt(ini)} --> {_ts_srt(fin)}\n{prefijo}{texto}\n')
    return '\n'.join(bloques) + ('\n' if bloques else '')


def _turnos(segments: list[dict[str, Any]]) -> list[tuple[str, str]]:
    """Agrupa segmentos consecutivos del mismo hablante → [(hablante, texto)]."""
    turnos: list[tuple[str, str]] = []
    actual_h: str | None = None
    actual_t: list[str] = []
    for seg in segments:
        texto = ' '.join(str(seg.get('text') or '').split())
        if not texto:
            continue
        bruto = seg.get('speaker')
        h = 'SIN_HABLANTE' if bruto in _HABLANTES_NULOS else str(bruto)
        if h != actual_h and actual_t:
            turnos.append((actual_h or 'SIN_HABLANTE', ' '.join(actual_t)))
            actual_t = []
        actual_h = h
        actual_t.append(texto)
    if actual_t:
        turnos.append((actual_h or 'SIN_HABLANTE', ' '.join(actual_t)))
    return turnos


def render_full_llm(segments: list[dict[str, Any]], metadata: dict[str, Any],
                    source_rel: str) -> str:
    """Texto corrido por turnos de hablante, sin timestamps — para pegar en un LLM."""
    partes = [f'Transcripción de: {source_rel}',
              f'Idioma: {metadata.get("language_detected") or metadata.get("language") or "?"} '
              f'· Duración: {format_ts(metadata.get("duration_seconds") or 0)} '
              f'· Palabras: {metadata.get("total_words", "?")}', '']
    turnos = _turnos(segments)
    if not turnos:
        partes.append('(sin voz detectada)')
    for hablante, texto in turnos:
        partes.append(f'{hablante}: {texto}')
        partes.append('')
    return '\n'.join(partes).rstrip('\n') + '\n'


def render_plano(segments: list[dict[str, Any]]) -> str:
    """Una línea por segmento `[HABLANTE] texto`, sin timestamps."""
    lineas = []
    for seg in segments:
        texto = ' '.join(str(seg.get('text') or '').split())
        if not texto:
            continue
        bruto = seg.get('speaker')
        h = 'SIN_HABLANTE' if bruto in _HABLANTES_NULOS else str(bruto)
        lineas.append(f'[{h}] {texto}')
    return '\n'.join(lineas) + ('\n' if lineas else '(sin voz detectada)\n')


def render_splits(texto_full: str, palabras_por_split: int) -> list[str]:
    """Trozos de ~N palabras respetando líneas completas, con cabecera de parte."""
    lineas = texto_full.splitlines()
    trozos: list[list[str]] = [[]]
    cuenta = 0
    for linea in lineas:
        n = len(linea.split())
        if cuenta + n > palabras_por_split and cuenta > 0:
            trozos.append([])
            cuenta = 0
        trozos[-1].append(linea)
        cuenta += n
    total = len(trozos)
    return [f'# parte {i}/{total}\n' + '\n'.join(t).strip() + '\n'
            for i, t in enumerate(trozos, 1) if any(l.strip() for l in t)] or \
        [f'# parte 1/1\n{texto_full}']


# ── JSON maestro (habilita republicar/renombrar sin GPU) ─────────────
def ruta_json_maestro(source_id: str) -> Path:
    return INTERMEDIOS_ROOT / f'{source_id}.json.gz'


def guardar_json_maestro(metadata: dict[str, Any], info: SourceInfo,
                         signature: str, motor_id: str) -> dict[str, Any] | None:
    """Persiste la metadata CRUDA del motor (segmentos sin `words`, comprimida)."""
    if not GUARDAR_JSON_MAESTRO:
        return None
    maestro = {k: v for k, v in metadata.items() if k != 'segments'}
    maestro['segments'] = [
        {'start': s.get('start'), 'end': s.get('end'),
         'text': s.get('text'), 'speaker': s.get('speaker')}
        for s in (metadata.get('segments') or [])]
    maestro['_v5'] = {'words_stripped': True, 'source_rel': info.relative.as_posix(),
                      'source_id': info.source_id, 'content_signature': signature,
                      'motor_profile_id': motor_id, 'creado': utc_now()}
    destino = ruta_json_maestro(info.source_id)
    tam, sha = atomic_write_gz_json(destino, maestro)
    return {'relative_path': destino.relative_to(OUTPUT_ROOT).as_posix(),
            'size_bytes': tam, 'sha256': sha}


def leer_json_maestro(source_id: str) -> dict[str, Any] | None:
    return read_gz_json(ruta_json_maestro(source_id))


# ── Perfil de PRESENTACIÓN (separado del perfil de MOTOR) ────────────
def presentacion_global() -> dict[str, Any]:
    """Parámetros que solo afectan el RENDER: cambiarlos NUNCA gasta GPU.

    `plantilla`/`espejo` se registran pero NO entran en el id: los nombres ya
    publicados son estables (aplican solo a archivos nuevos).
    """
    nucleo = {
        'schema': OUTPUT_SCHEMA,
        'encabezado': bool(INCLUIR_ENCABEZADO),
        'timestamp_ms': bool(TIMESTAMP_MILISEGUNDOS),
        'muletillas_md': str(MULETILLAS_MD),
        'gap_md_s': float(GAP_BLOQUE_MD_S),
        'split_palabras': int(SPLIT_PALABRAS),
        'entregables': {'txt': bool(SALIDA_TXT_TIMESTAMPS), 'md': bool(SALIDA_TRANSCRIPT_MD),
                        'srt': bool(SALIDA_SRT), 'json': bool(SALIDA_JSON),
                        'full_llm': bool(SALIDA_FULL_LLM), 'splits': bool(SALIDA_SPLITS),
                        'plano': bool(SALIDA_TXT_PLANO)},
        'repara_huerfanos': bool(REPARAR_HABLANTES_HUERFANOS),
        'tolerancia_huerfano_s': float(TOLERANCIA_HUERFANO_S),
        'max_fraccion_sin_hablante': float(MAX_FRACCION_SIN_HABLANTE),
        'acepta_sin_voz': bool(ACEPTAR_AUDIO_SIN_VOZ),
    }
    return {'id': canonical_hash(nucleo), **nucleo,
            'plantilla': PLANTILLA_NOMBRE, 'espejo': bool(ESPEJO_SUBCARPETAS)}


PRESENTACION_GLOBAL = presentacion_global()


def presentacion_para(source_id: str) -> dict[str, Any]:
    """Perfil de presentación por archivo (incluye sus renombres persistentes)."""
    r_sha = rename_sha_de(cargar_renombres(source_id))
    pid = canonical_hash({'global': PRESENTACION_GLOBAL['id'], 'rename': r_sha})
    return {'id': pid, 'global_id': PRESENTACION_GLOBAL['id'], 'rename_sha': r_sha}


def rutas_entregables(final_txt: Path) -> dict[str, Path]:
    """Mapa entregable_activo → ruta final (excepto splits, que son N archivos)."""
    base = final_txt.with_suffix('')
    mapa = {}
    if SALIDA_TRANSCRIPT_MD:
        mapa['md'] = base.with_name(base.name + '.transcript.md')
    if SALIDA_SRT:
        mapa['srt'] = base.with_name(base.name + '.srt')
    if SALIDA_JSON:
        mapa['json'] = base.with_name(base.name + '.json')
    if SALIDA_FULL_LLM:
        mapa['full_llm'] = base.with_name(base.name + '.full_for_llm.txt')
    if SALIDA_TXT_PLANO:
        mapa['plano'] = base.with_name(base.name + '.plano.txt')
    return mapa


def _escribir_extra(destino: Path, contenido: str) -> dict[str, Any]:
    atomic_write_text(destino, contenido)
    return {'relative_path': destino.relative_to(OUTPUT_ROOT).as_posix(),
            'size_bytes': destino.stat().st_size, 'sha256': hash_file(destino, 'full')}


def publicar_extras(final_txt: Path, segments: list[dict[str, Any]],
                    metadata: dict[str, Any], source_rel: str) -> list[dict[str, Any]]:
    """Escribe todos los entregables secundarios activos (idempotente, atómico)."""
    base = final_txt.with_suffix('')
    rutas = rutas_entregables(final_txt)
    extras: list[dict[str, Any]] = []

    if 'srt' in rutas:
        extras.append({'tipo': 'srt', **_escribir_extra(rutas['srt'], render_srt(segments))})
    if 'plano' in rutas:
        extras.append({'tipo': 'plano', **_escribir_extra(rutas['plano'], render_plano(segments))})
    texto_full = None
    if 'full_llm' in rutas or SALIDA_SPLITS:
        texto_full = render_full_llm(segments, metadata, source_rel)
    if 'full_llm' in rutas:
        extras.append({'tipo': 'full_llm', **_escribir_extra(rutas['full_llm'], texto_full)})
    if SALIDA_SPLITS:
        partes = render_splits(texto_full, int(SPLIT_PALABRAS))
        for viejo in final_txt.parent.glob(base.name + '.parte_*.txt'):
            with contextlib.suppress(OSError):
                viejo.unlink()          # limpia splits obsoletos de corridas previas
        for i, parte in enumerate(partes, 1):
            destino = base.with_name(f'{base.name}.parte_{i:02d}.txt')
            extras.append({'tipo': f'split_{i:02d}', **_escribir_extra(destino, parte)})
    if 'json' in rutas:
        maestro = {k: v for k, v in metadata.items() if k != 'segments'}
        maestro['segments'] = [{'start': s.get('start'), 'end': s.get('end'),
                                'text': s.get('text'), 'speaker': s.get('speaker')}
                               for s in (metadata.get('segments') or [])]
        contenido = json.dumps(maestro, ensure_ascii=False, indent=1, default=str)
        extras.append({'tipo': 'json', **_escribir_extra(rutas['json'], contenido)})
    if 'md' in rutas:
        try:
            from speakerscribe.output import generate_transcript_md
            tmp_md = rutas['md'].with_name(rutas['md'].name + f'.tmp.{RUN_ID}')
            generate_transcript_md(list(segments), tmp_md, dict(metadata),
                                   gap_max_s=float(GAP_BLOQUE_MD_S),
                                   remove_fillers=MULETILLAS_MD)
            os.replace(tmp_md, rutas['md'])
            extras.append({'tipo': 'md',
                           'relative_path': rutas['md'].relative_to(OUTPUT_ROOT).as_posix(),
                           'size_bytes': rutas['md'].stat().st_size,
                           'sha256': hash_file(rutas['md'], 'full')})
        except Exception as exc:
            print(f'   ⚠️ .transcript.md no generado para {source_rel}: {sanitize_error(exc)[:140]}')
    return extras


def extras_activos_presentes(final_txt: Path) -> bool:
    """True si TODOS los entregables activos existen junto al .txt confirmado."""
    try:
        for ruta in rutas_entregables(final_txt).values():
            if not ruta.is_file() or ruta.stat().st_size == 0:
                return False
        if SALIDA_SPLITS:
            base = final_txt.with_suffix('')
            if not (base.with_name(base.name + '.parte_01.txt')).is_file():
                return False
        return True
    except OSError:
        return False


def estado_desde_maestro(metadata: dict[str, Any]) -> tuple[str, str | None]:
    """Recalcula la categoría de calidad desde el JSON maestro (sin GPU).

    Se usa al republicar: el contenido ya fue aceptado una vez, así que aquí
    nunca se rechaza — solo se clasifica y se marca honestamente.
    """
    if metadata.get('diarization_failed') or not metadata.get('diarization_enabled'):
        detalle = re.sub(r'^\w+Error:\s*', '', str(metadata.get('diarization_error') or ''))
        return 'degradado_sin_diarizacion', (detalle or 'diarización no disponible')[:300]
    criticos = [str(f) for f in (metadata.get('quality_flags') or [])
                if str(f).startswith('[CRITICAL]')]
    if RECHAZAR_CALIDAD_CRITICA and criticos:
        return 'publicado_con_flags_criticos', ('; '.join(criticos))[:300]
    return 'ok', None


def publicar_desde_metadata(info: SourceInfo, output_relative: Path,
                            metadata: dict[str, Any], estado: str,
                            motivo: str | None, job_id: str,
                            signature: str, *, valvula: bool = True) -> dict[str, Any]:
    """Render + escritura de TODOS los entregables. El .txt queda como `.part`
    listo para el commit `prepared→completed`; los extras son idempotentes.

    Args:
        valvula: si el render estricto falla, True degrada a
            'publicado_con_flags_criticos' y publica MARCADO; False re-lanza
            el ValueError (el llamador decide rechazar).

    Returns:
        Dict con part/size/sha del .txt, extras, estado final y reparación.

    Raises:
        ValueError: solo con valvula=False y render estricto fallido.
    """
    final_txt = safe_output_path(output_relative)
    segs, repair, _r_sha = preparar_segmentos(metadata, info.source_id)
    try:
        texto = render_transcript(segs, metadata, info.relative.as_posix(),
                                  estado, motivo, repair)
    except ValueError as exc:
        if not valvula:
            raise
        estado, motivo = 'publicado_con_flags_criticos', str(exc)[:300]
        texto = render_transcript(segs, metadata, info.relative.as_posix(),
                                  estado, motivo, repair)
    extras = publicar_extras(final_txt, segs, metadata, info.relative.as_posix())
    part, tam, sha = write_complete_part(final_txt, texto, job_id)
    return {'part': part, 'size': tam, 'sha256': sha, 'extras': extras,
            'estado': estado, 'motivo': motivo, 'repair': repair,
            'final_txt': final_txt}


print('🎨 Render v5 cargado: reparación de huérfanos, renombres persistentes, '
      '7 entregables, JSON maestro y perfil de presentación.')

## 🟢 6 · Motor, planificador y proceso (solo definiciones — no edites)

Configura el motor `speakerscribe`, separa el perfil de **motor** (lo que gasta GPU) del de **presentación** (lo que se republica en CPU), pre-verifica Hugging Face **antes** de descargar nada, planifica en bloques incrementales (el primer `.txt` sale en minutos) y define `process_job` con la válvula anti-pérdida.

In [ ]:
# ===== 🟢 CELDA 6 — MOTOR, PLANIFICADOR Y PROCESO (solo definiciones) =====
def none_if_zero(v: int) -> int | None:
    v = int(v)
    return None if v <= 0 else v


def make_engine_config():
    """TranscriptionConfig del motor. El notebook es dueño del estado:
    `enable_runs_db=False` + `force_reprocess=True` (la idempotencia vive en
    el journal v5, no en el ledger interno del paquete)."""
    from speakerscribe import TranscriptionConfig

    return TranscriptionConfig(
        model=MODELO, language=(IDIOMA.strip().lower() or None),
        device='cuda', compute_type='float16',
        beam_size=int(BEAM_SIZE), batch_size=int(BATCH_SIZE),
        initial_prompt=(GLOSARIO_GLOBAL.strip() or None),
        use_vad=True, vad_min_silence_ms=int(VAD_MIN_SILENCE_MS),
        condition_on_previous_text=(not ANTI_ALUCINACION),
        enable_diarization=True, speaker_assignment='word',
        diarization_model='pyannote/speaker-diarization-community-1',
        num_speakers=none_if_zero(NUM_HABLANTES),
        min_speakers=none_if_zero(MIN_HABLANTES),
        max_speakers=none_if_zero(MAX_HABLANTES),
        word_timestamps=False, generate_transcript_md=False,
        remove_fillers='off', produce_unified_for_llm=False, streaming_jsonl=False,
        long_audio_threshold_min=0, extract_temp_wav=True, delete_temp_wav=True,
        delete_chunk_wavs=True, evaluate_quality=True, auto_retry_on_critical=True,
        enable_runs_db=False, force_reprocess=True, hash_mode='fast',
    )


def motor_profile(config: Any, prompt_digest: str, package_version: str) -> dict[str, Any]:
    """Identidad SEMÁNTICA de la transcripción (lo que exige GPU si cambia).

    Deliberadamente NO incluye: versiones del entorno (torch, CUDA — una
    actualización rutinaria de Colab no invalida tu archivo), la versión del
    notebook, ni NADA de presentación (formato, plantilla, entregables).
    """
    perfil = {
        'engine': 'speakerscribe', 'speakerscribe_version': package_version,
        'asr_model': config.model, 'language': config.language,
        'device': config.device, 'compute_type': config.compute_type,
        'beam_size': config.beam_size, 'batch_size': config.batch_size,
        'use_vad': config.use_vad, 'vad_min_silence_ms': config.vad_min_silence_ms,
        'condition_on_previous_text': config.condition_on_previous_text,
        'speaker_assignment': config.speaker_assignment,
        'diarization_model': config.diarization_model,
        'num_speakers': config.num_speakers, 'min_speakers': config.min_speakers,
        'max_speakers': config.max_speakers,
        'prompt_sha256': prompt_digest,
    }
    return {'id': canonical_hash(perfil), **perfil}


def verificar_acceso_hf(config: Any) -> None:
    """Corta ANTES de descargar modelos si el token es inválido o el modelo
    gated no fue aceptado — el fallo más caro posible."""
    token = config.resolve_hf_token()
    if not token:
        raise RuntimeError(
            'No se encontró HF_TOKEN.\n'
            '  1. Token tipo Read (no fine-grained): https://huggingface.co/settings/tokens\n'
            f'  2. Acepta las condiciones: https://huggingface.co/{config.diarization_model}\n'
            '  3. Guárdalo en Secrets de Colab (🔑) como HF_TOKEN, con acceso al notebook.')
    try:
        from huggingface_hub import HfApi
        try:
            from huggingface_hub.errors import GatedRepoError, HfHubHTTPError
        except Exception:
            from huggingface_hub.utils import GatedRepoError, HfHubHTTPError  # type: ignore
        api = HfApi(token=token)
        try:
            api.whoami()
        except Exception as exc:
            raise RuntimeError('HF_TOKEN existe pero Hugging Face lo rechaza '
                               f'(¿revocado/mal copiado?): {sanitize_error(exc)}') from exc
        try:
            api.model_info(config.diarization_model)
        except GatedRepoError as exc:
            raise RuntimeError(
                f'El token no tiene acceso a {config.diarization_model}: acepta las '
                f'condiciones del modelo y reintenta. {sanitize_error(exc)}') from exc
        except HfHubHTTPError as exc:
            print(f'Aviso: pre-verificación del modelo gated no concluyente '
                  f'({sanitize_error(exc)}); se validará al cargar la diarización.')
    except RuntimeError:
        raise
    except Exception as exc:
        print(f'Aviso: pre-verificación de Hugging Face omitida ({sanitize_error(exc)}).')


def preflight_storage() -> None:
    """Validación sin GPU ni token: una sesión sin pendientes no debe exigirlos."""
    if not INPUT_ROOT.is_dir():
        raise FileNotFoundError(f'No existe la entrada: {INPUT_ROOT}')
    if IN_COLAB and (str(OUTPUT_ROOT).startswith('/content/')
                     and not str(OUTPUT_ROOT).startswith('/content/drive/')):
        raise RuntimeError('RUTA_SALIDA está en el disco efímero de Colab: usa /content/drive/…')
    if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
        raise RuntimeError('ffmpeg/ffprobe no disponibles — reinicia la sesión de Colab.')
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    for d in (STATE_ROOT, GLOBAL_DIAR_CACHE, INTERMEDIOS_ROOT, RENOMBRES_ROOT,
              FAIL_LOGS_ROOT, JOBS_ROOT, ALIAS_ROOT):
        d.mkdir(parents=True, exist_ok=True)


def preflight_gpu(config: Any, jobs: list) -> dict[str, Any]:
    """Se ejecuta perezosamente: solo cuando existe trabajo real. Se adapta a
    cualquier GPU usando los mínimos de VRAM por modelo del propio paquete."""
    import importlib
    import importlib.metadata as metadata
    import torch
    import speakerscribe
    from speakerscribe.config import MIN_VRAM_BY_MODEL

    if not torch.cuda.is_available():
        raise RuntimeError('No hay GPU CUDA. Entorno de ejecución → Cambiar tipo de '
                           'entorno → T4 GPU (o superior), y vuelve a «Ejecutar todo».')
    verificar_acceso_hf(config)
    for mod in ('faster_whisper', 'ctranslate2', 'pyannote.audio'):
        try:
            importlib.import_module(mod)
        except Exception as exc:
            raise RuntimeError(f'El stack no puede importar {mod}: {exc}') from exc
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    req = MIN_VRAM_BY_MODEL.get(config.model, 5.0) + 1.5   # + margen de diarización
    if vram < req:
        raise RuntimeError(f'VRAM insuficiente: {vram:.1f} GB < ~{req:.1f} GB para '
                           f'{config.model} + diarización. Usa una GPU mayor o un modelo menor.')
    mayor = max((((j.duration_seconds or 0.0) * 32_000) or j.source.size_bytes * 4
                 for j in jobs), default=0.0)
    requerido = max(500_000_000, int(max(mayor, max((j.source.size_bytes * 4 for j in jobs),
                                                    default=0)) * 1.25))
    libre = shutil.disk_usage(SCRATCH_ROOT).free
    if jobs and libre < requerido:
        raise RuntimeError(f'Scratch insuficiente: {libre / 1e9:.1f} GB libres, '
                           f'~{requerido / 1e9:.1f} GB requeridos.')
    versiones = {'speakerscribe': speakerscribe.__version__, 'torch': torch.__version__}
    for p in ('faster-whisper', 'ctranslate2', 'pyannote.audio', 'torchaudio', 'transformers'):
        with contextlib.suppress(metadata.PackageNotFoundError):
            versiones[p] = metadata.version(p)
    return {'gpu': gpu, 'vram_gb': round(vram, 2),
            'scratch_libre_gb': round(libre / 1e9, 2), 'versiones': versiones}


def latest_success_record(records: list[dict[str, Any]]) -> dict[str, Any] | None:
    for rec in reversed(records):
        if rec.get('event') in {'completed', 'reused', 'republished'}:
            return rec
    return None


def valid_success(records: list[dict[str, Any]]) -> dict[str, Any] | None:
    for rec in reversed(records):
        if output_record_is_valid(rec):
            return rec
    return None


def create_planner_state(sources: list[SourceInfo],
                         colisiones: set[tuple[str, str]],
                         events: list[dict[str, Any]] | None = None) -> dict[str, Any]:
    """Índices, mapa de nombres ESTABLE y agenda de auditoría — una sola vez.

    Estabilidad: un nombre ya publicado se conserva aunque cambien la
    plantilla o aparezcan homónimos nuevos. Los archivos nuevos usan la
    plantilla actual con escalera anti-colisión.
    """
    indices = build_indices(read_events() if events is None else events)

    def ckey(p: Path) -> str:
        return nfc(p.as_posix()).casefold()

    usados: dict[str, str] = {}
    output_map: dict[str, Path] = {}
    actuales = {s.source_id for s in sources}
    # 1) Estabilidad: los nombres ya publicados se conservan.
    for sid, records in sorted(indices['ok_por_fuente'].items()):
        prior = latest_success_record(records)
        rel = (prior.get('output') or {}).get('relative_path') if prior else None
        if rel and output_path_from_record(prior) is not None:
            cand = Path(rel)
            k = ckey(cand)
            if k not in usados or usados[k] == sid:
                usados[k] = sid
                if sid in actuales:
                    output_map[sid] = cand
    # 2) Nuevos: espejo + plantilla + escalera anti-colisión residual.
    for s in sources:
        if s.source_id in output_map:
            continue
        cand = output_relative_for(s, colisiones)
        k = ckey(cand)
        if k in usados and usados[k] != s.source_id:
            base = cand
            for n in (10, 16, 24, 32, 64):
                cand = base.with_name(f'{base.stem}~{s.source_id[:n]}{base.suffix}')
                k = ckey(cand)
                if k not in usados or usados[k] == s.source_id:
                    break
            else:
                raise RuntimeError(f'Sin salida única para {s.relative}')
        usados[k] = s.source_id
        output_map[s.source_id] = cand

    def confirmado(s: SourceInfo) -> bool:
        prior = latest_success_record(indices['ok_por_fuente'].get(s.source_id, []))
        return bool(prior and source_stat_matches(prior, s)
                    and output_record_stat_is_valid(prior))

    orden = sorted(sources, key=lambda i: (nfc(i.relative.as_posix()).casefold(), i.source_id))
    cambiados = [s for s in orden if not confirmado(s)]
    estables = [s for s in orden if confirmado(s)]
    try:
        cursor = json.loads(AUDIT_CURSOR_PATH.read_text(encoding='utf-8'))
    except Exception:
        cursor = {}
    ids_estables = [s.source_id for s in estables]
    inicio = ((ids_estables.index(cursor.get('last_source_id')) + 1) % len(ids_estables)
              if cursor.get('last_source_id') in ids_estables else 0) if ids_estables else 0
    rotados = estables[inicio:] + estables[:inicio]
    auditar: list[SourceInfo] = []
    bytes_aud = 0
    tope_bytes = int(float(MAX_GB_AUDITORIA_POR_SESION) * 1e9)
    for s in rotados:
        if len(auditar) >= int(AUDITORIAS_ESTRICTAS_POR_SESION):
            break
        if auditar and bytes_aud + s.size_bytes > tope_bytes:
            break
        auditar.append(s)
        bytes_aud += s.size_bytes
    strict_ids = {s.source_id for s in auditar}
    resto = [s for s in estables if s.source_id not in strict_ids]
    return {'indices': indices, 'output_map': output_map,
            'ordered': cambiados + auditar + resto, 'strict_ids': strict_ids,
            'audit_n': len(auditar), 'audit_gb': round(bytes_aud / 1e9, 2)}


def _politica_perfil(rel: str, previo: dict[str, Any], nuevo: dict[str, Any],
                     conteo: dict[str, Any], fast: bool) -> bool:
    """True ⇒ conservar y saltar; False ⇒ reprocesar; 'detener' ⇒ excepción."""
    conteo['perfil_cambiado'] += 1
    if POLITICA_CAMBIO_PERFIL == 'detener':
        raise RuntimeError(f'Cambió el perfil de MOTOR para {rel}. Elige explícitamente '
                           'conservar_y_reportar o reprocesar en POLITICA_CAMBIO_PERFIL.')
    if POLITICA_CAMBIO_PERFIL == 'conservar_y_reportar':
        append_event('profile_change_deferred', source=previo.get('source'),
                     previous_profile=previo.get('profile'), requested_profile=nuevo,
                     output=previo.get('output'), fast_path=fast)
        conteo['ya_listos'] += 1
        return True
    return False


def _master_meta_actual(source_id: str) -> dict[str, Any] | None:
    """Metadatos (ruta/tamaño/sha) del JSON maestro existente en disco."""
    ruta = ruta_json_maestro(source_id)
    try:
        if not ruta.is_file():
            return None
        return {'relative_path': ruta.relative_to(OUTPUT_ROOT).as_posix(),
                'size_bytes': ruta.stat().st_size, 'sha256': hash_file(ruta, 'full')}
    except OSError:
        return None


def commit_publicacion(job: Job, pub: dict[str, Any], evento: str,
                       presentacion: dict[str, Any],
                       master_meta: dict[str, Any] | None,
                       **extra: Any) -> dict[str, Any]:
    """Commit transaccional: `prepared` (con .part) → promoción → evento final."""
    salida = {'relative_path': job.output_relative.as_posix(),
              'part_relative_path': pub['part'].relative_to(OUTPUT_ROOT).as_posix(),
              'size_bytes': pub['size'], 'sha256': pub['sha256'],
              'encoding': 'utf-8', 'schema': OUTPUT_SCHEMA}
    campos: dict[str, Any] = dict(
        job_id=job.job_id, source=source_payload(job.source, job.signature),
        profile=job.profile, presentacion=presentacion,
        extras=pub['extras'], master_json=master_meta, quality=pub['estado'])
    if pub.get('motivo'):
        campos['degraded_reason'] = pub['motivo']
    if pub.get('repair') and pub['repair'].get('adoptados'):
        campos['speaker_repair'] = pub['repair']
    campos.update(extra)
    append_event('prepared', output=salida, **campos)
    promote_part(pub['part'], pub['final_txt'])
    return append_event(evento,
                        output={k: v for k, v in salida.items()
                                if k != 'part_relative_path'},
                        **campos)


def republicar_fuente(info: SourceInfo, prior: dict[str, Any],
                      evento: str = 'republished') -> dict[str, Any] | None:
    """Re-renderiza TODOS los entregables desde el JSON maestro — sin GPU.

    Se usa cuando cambia la presentación (formato, entregables, renombres,
    encabezado…) o cuando falta un entregable activo. Conserva el nombre de
    salida histórico (estabilidad).

    Returns:
        El evento final si se republicó; None si no hay JSON maestro.
    """
    maestro = leer_json_maestro(info.source_id)
    if maestro is None:
        return None
    firma = (prior.get('source') or {}).get('content_signature') or ''
    rel_salida = Path((prior.get('output') or {}).get('relative_path'))
    perfil_motor = prior.get('profile') or {'id': maestro.get('_v5', {}).get('motor_profile_id', '?')}
    presentacion = presentacion_para(info.source_id)
    job_id = canonical_hash({'content': firma or info.source_id,
                             'profile': perfil_motor.get('id'),
                             'output': rel_salida.as_posix(),
                             'presentacion': presentacion['id']})
    job = Job(info, firma, str(perfil_motor.get('id')), perfil_motor, job_id,
              rel_salida, maestro.get('duration_seconds'))
    estado, motivo = estado_desde_maestro(maestro)
    pub = publicar_desde_metadata(info, rel_salida, maestro, estado, motivo,
                                  job_id, firma, valvula=True)
    return commit_publicacion(job, pub, evento, presentacion,
                              _master_meta_actual(info.source_id))


def reutilizar_por_contenido(info: SourceInfo, firma: str, perfil: dict[str, Any],
                             rel_salida: Path, donante: dict[str, Any]) -> dict[str, Any] | None:
    """Dedup sin GPU: copia el JSON maestro del donante (misma identidad fuerte
    de contenido + mismo perfil de motor) y re-renderiza para ESTA fuente con
    su propia presentación y nombre. Devuelve None si el donante no tiene maestro."""
    donante_sid = (donante.get('source') or {}).get('id')
    rel_master = (donante.get('master_json') or {}).get('relative_path')
    origen = (OUTPUT_ROOT / rel_master) if rel_master else \
        (ruta_json_maestro(donante_sid) if donante_sid else None)
    if origen is None or not origen.is_file():
        return None
    maestro = read_gz_json(origen)
    if maestro is None:
        return None
    if GUARDAR_JSON_MAESTRO:
        destino = ruta_json_maestro(info.source_id)
        if origen.resolve() != destino.resolve():
            tmp = destino.with_name(destino.name + f'.tmp.{RUN_ID}')
            destino.parent.mkdir(parents=True, exist_ok=True)
            try:
                shutil.copy2(origen, tmp)
                os.replace(tmp, destino)
            finally:
                with contextlib.suppress(OSError):
                    if tmp.exists():
                        tmp.unlink()
    presentacion = presentacion_para(info.source_id)
    job_id = canonical_hash({'content': firma, 'profile': perfil['id'],
                             'output': rel_salida.as_posix(),
                             'presentacion': presentacion['id']})
    job = Job(info, firma, perfil['id'], perfil, job_id, rel_salida,
              maestro.get('duration_seconds'))
    estado, motivo = estado_desde_maestro(maestro)
    pub = publicar_desde_metadata(info, rel_salida, maestro, estado, motivo,
                                  job_id, firma, valvula=True)
    return commit_publicacion(job, pub, 'reused', presentacion,
                              _master_meta_actual(info.source_id))


def plan_block(config: Any, package_version: str, state: dict[str, Any],
               candidates: list[SourceInfo],
               deadline_reached) -> tuple[list[Job], dict[str, Any]]:
    """Planifica un bloque pequeño: el primer TXT sale en minutos, no al final
    de hashear todo el histórico. Aquí también se resuelven, SIN GPU, las
    republicaciones por cambio de presentación y la dedup por contenido."""
    indices = state['indices']
    output_map = state['output_map']
    strict_ids = state['strict_ids']
    jobs: list[Job] = []
    conteo = {'examinados': 0, 'ya_listos': 0, 'republicados': 0, 'reusados': 0,
              'invalidos': 0, 'pospuestos': 0, 'bloqueados': 0,
              'probe_reintentable': 0, 'perfil_cambiado': 0, 'sin_maestro': 0,
              'diferidos_presupuesto': 0}

    def _indexar_exito(rec: dict[str, Any], sid: str, firma: str | None,
                       perfil_id: str | None) -> None:
        indices['ok_por_fuente'].setdefault(sid, []).append(rec)
        indices['sucias'].discard(sid)
        if firma and not str(firma).startswith('fast:') and perfil_id:
            indices['ok_por_contenido_perfil'].setdefault((firma, perfil_id), []).append(rec)

    def _refrescar_o_conservar(original: SourceInfo, prior: dict[str, Any],
                               fast: bool) -> None:
        """Presentación desactualizada o entregable faltante → republicar en CPU."""
        try:
            rec = republicar_fuente(original, prior)
        except Exception as exc:
            print(f'   ⚠️ Republicación falló para {original.relative.as_posix()}: '
                  f'{sanitize_error(exc)[:140]} (se conserva la salida previa)')
            conteo['ya_listos'] += 1
            return
        if rec is None:
            conteo['sin_maestro'] += 1
            conteo['ya_listos'] += 1
            return
        firma_prev = (prior.get('source') or {}).get('content_signature')
        _indexar_exito(rec, original.source_id, firma_prev,
                       (prior.get('profile') or {}).get('id'))
        conteo['republicados'] += 1

    for original in candidates:
        if deadline_reached():
            conteo['diferidos_presupuesto'] += 1
            continue
        conteo['examinados'] += 1

        try:
            edad = time.time() - original.path.stat().st_mtime
        except OSError as exc:
            append_event('probe_retryable', source=source_payload(original),
                         error=sanitize_error(exc), stage='stat')
            conteo['probe_reintentable'] += 1
            continue
        if edad < SEGUNDOS_ESTABILIDAD_ARCHIVO:
            append_event('source_changed', source=source_payload(original),
                         error=f'modificado hace {edad:.1f} s; se pospone')
            conteo['pospuestos'] += 1
            continue

        # ── Fast-path de confirmados: sin releer el audio (auditados y sucios NO) ──
        if (not FORZAR_REPROCESO and original.source_id not in strict_ids
                and original.source_id not in indices['sucias']):
            prior = latest_success_record(indices['ok_por_fuente'].get(original.source_id, []))
            if prior and source_stat_matches(prior, original):
                try:
                    perfil_f = motor_profile(config, prompt_hash(original), package_version)
                except OSError as exc:
                    append_event('probe_retryable', source=source_payload(original),
                                 error=sanitize_error(exc), stage='fast_prompt')
                    conteo['probe_reintentable'] += 1
                    continue
                if output_record_is_valid(prior):
                    if (prior.get('profile') or {}).get('id') == perfil_f['id']:
                        pres = presentacion_para(original.source_id)
                        final_txt = output_path_from_record(prior)
                        al_dia = ((prior.get('presentacion') or {}).get('id') == pres['id']
                                  and final_txt is not None
                                  and extras_activos_presentes(final_txt))
                        if al_dia:
                            conteo['ya_listos'] += 1
                        else:
                            _refrescar_o_conservar(original, prior, fast=True)
                        continue
                    if _politica_perfil(original.relative.as_posix(), prior, perfil_f,
                                        conteo, fast=True):
                        continue
                    # 'reprocesar' cae al camino completo ↓

        prior_invalido = indices['invalidos'].get(original.source_id)
        if (not FORZAR_REPROCESO and prior_invalido
                and source_stat_matches(prior_invalido, original)):
            conteo['invalidos'] += 1
            continue

        estado_probe, duracion, err_probe = ffprobe_audio(original.path)
        if estado_probe != 'ok':
            if estado_probe == 'invalid':
                append_event('invalid_media', source=source_payload(original),
                             error=err_probe or 'sin audio', stage='ffprobe')
                conteo['invalidos'] += 1
            else:
                append_event('probe_retryable', source=source_payload(original),
                             error=err_probe or 'ffprobe', stage='ffprobe')
                conteo['probe_reintentable'] += 1
            continue

        try:
            firma, info = stable_signature(original)
            perfil = motor_profile(config, prompt_hash(info), package_version)
        except SourceChangingError as exc:
            append_event('source_changed', source=source_payload(original),
                         error=sanitize_error(exc))
            conteo['pospuestos'] += 1
            continue
        except OSError as exc:
            append_event('probe_retryable', source=source_payload(original),
                         error=sanitize_error(exc), stage='hash')
            conteo['probe_reintentable'] += 1
            continue

        rel_salida = output_map[info.source_id]
        presentacion = presentacion_para(info.source_id)
        job_id = canonical_hash({'content': firma, 'profile': perfil['id'],
                                 'output': rel_salida.as_posix()})
        job = Job(info, firma, perfil['id'], perfil, job_id, rel_salida, duracion)

        if not FORZAR_REPROCESO:
            prior = valid_success(indices['ok_por_fuente'].get(info.source_id, []))
            if prior:
                firma_previa = (prior.get('source') or {}).get('content_signature')
                perfil_previo = (prior.get('profile') or {}).get('id')
                if firma_previa and firma_previa != firma:
                    append_event('content_change_detected', job_id=job.job_id,
                                 source=source_payload(info, firma),
                                 previous_content_signature=firma_previa, profile=perfil)
                    indices['sucias'].add(info.source_id)
                if firma_previa == firma and perfil_previo == perfil['id']:
                    if info.source_id in indices['sucias']:
                        append_event('content_change_resolved',
                                     source=source_payload(info, firma), profile=perfil)
                        indices['sucias'].discard(info.source_id)
                    final_txt = output_path_from_record(prior)
                    al_dia = ((prior.get('presentacion') or {}).get('id') == presentacion['id']
                              and final_txt is not None
                              and extras_activos_presentes(final_txt))
                    if al_dia:
                        conteo['ya_listos'] += 1
                    else:
                        _refrescar_o_conservar(info, prior, fast=False)
                    continue
                if firma_previa == firma and perfil_previo != perfil['id']:
                    if _politica_perfil(info.relative.as_posix(), prior, perfil,
                                        conteo, fast=False):
                        continue

            # Dedup por contenido (solo identidad FUERTE): otra fuente con el
            # mismo audio y el mismo perfil de motor dona su JSON maestro.
            donante = valid_success(
                indices['ok_por_contenido_perfil'].get((firma, perfil['id']), []))
            if donante and (donante.get('source') or {}).get('id') != info.source_id:
                try:
                    rec = reutilizar_por_contenido(info, firma, perfil, rel_salida, donante)
                except Exception as exc:
                    print(f'   (dedup no aplicable para {info.relative.as_posix()}: '
                          f'{sanitize_error(exc)[:120]}; se procesa en GPU)')
                    rec = None
                if rec is not None:
                    _indexar_exito(rec, info.source_id, firma, perfil['id'])
                    conteo['reusados'] += 1
                    continue

        if (indices['intentos'].get(job_id, 0) >= MAX_INTENTOS_POR_ARCHIVO
                and not FORZAR_REPROCESO):
            conteo['bloqueados'] += 1
            continue
        jobs.append(job)
    return jobs, conteo


# ── Caché durable de diarización (sobrevive reintentos y sesiones) ───
def _diar_profile(config: Any) -> str:
    return canonical_hash({'cache_schema': 1, 'model': config.diarization_model,
                           'num': config.num_speakers, 'min': config.min_speakers,
                           'max': config.max_speakers})


def copy_diar_cache_in(job: Job, paths: Any, config: Any, media_stem: str) -> tuple[Path, Path]:
    from speakerscribe.diarization import diarization_params_hash

    local = paths.diar_cache / f'{media_stem}_{diarization_params_hash(config)}.diar.json'
    durable = GLOBAL_DIAR_CACHE / f'{job.signature}_{_diar_profile(config)}.diar.json'
    local.parent.mkdir(parents=True, exist_ok=True)
    with contextlib.suppress(FileNotFoundError):
        local.unlink()
    if durable.is_file() and not FORZAR_REPROCESO:
        try:
            json.loads(durable.read_text(encoding='utf-8'))
            shutil.copy2(durable, local)
        except Exception:
            with contextlib.suppress(OSError):
                os.replace(durable, durable.with_name(
                    durable.name + f'.invalid.{int(time.time())}'))
    return local, durable


def copy_diar_cache_out(local: Path, durable: Path) -> None:
    """Persiste SIEMPRE (también tras un intento fallido): con parámetros
    idénticos la diarización es determinista y el reintento solo paga ASR."""
    if not local.is_file():
        return
    try:
        json.loads(local.read_text(encoding='utf-8'))
    except Exception:
        return
    durable.parent.mkdir(parents=True, exist_ok=True)
    tmp = durable.with_name(durable.name + f'.tmp.{RUN_ID}')
    try:
        shutil.copy2(local, tmp)
        os.replace(tmp, durable)
        fsync_directory(durable.parent)
    finally:
        with contextlib.suppress(OSError):
            if tmp.exists():
                tmp.unlink()


def make_local_alias(job: Job) -> tuple[Path, list[Path]]:
    """Symlink local con nombre único (evita colisiones y rutas raras del motor).
    Si el FS no admite symlinks, degrada con seguridad a la ruta original."""
    ALIAS_ROOT.mkdir(parents=True, exist_ok=True)
    suf = job.source.path.suffix.lower()
    suf = suf if re.fullmatch(r'\.[a-z0-9]{1,12}', suf) else '.media'
    alias = ALIAS_ROOT / f'src_{RUN_ID[:8]}_{job.job_id[:20]}{suf}'
    creados = [alias]
    try:
        with contextlib.suppress(FileNotFoundError):
            alias.unlink()
        os.symlink(job.source.path, alias)
        if job.source.prompt_path is not None:
            p = alias.with_suffix('.prompt.txt')
            with contextlib.suppress(FileNotFoundError):
                p.unlink()
            os.symlink(job.source.prompt_path, p)
            creados.append(p)
        return alias, creados
    except Exception:
        for c in creados:
            with contextlib.suppress(OSError):
                c.unlink()
        return job.source.path, []   # degradación segura: el workspace por-job aísla nombres


def _rescatar_logs_fallo(paths: Any, job: Job) -> None:
    if not DEPURAR_GUARDAR_LOGS_FALLO:
        return
    with contextlib.suppress(Exception):
        destino = FAIL_LOGS_ROOT / job.job_id[:12]
        destino.mkdir(parents=True, exist_ok=True)
        for log in sorted(paths.logs.glob('*.log'))[-2:]:
            shutil.copy2(log, destino / log.name)
        atomic_write_json(destino / 'contexto.json',
                          {'source': source_payload(job.source, job.signature),
                           'profile_id': job.profile_id, 'utc': utc_now()})


def process_job(job: Job, model: Any, diar_engine: Any, config: Any,
                intentos_previos: int) -> dict[str, Any]:
    """Un archivo → todos sus entregables confirmados + JSON maestro.

    En el ÚLTIMO intento (o desde la 1ª pasada si PUBLICAR_SI_CALIDAD_NO_MEJORA,
    porque el rechazo de calidad es determinista), la válvula publica el
    resultado MARCADO en lugar de dejarte sin nada.
    """
    from speakerscribe import WorkspacePaths, process_one

    job_ws = JOBS_ROOT / RUN_ID / job.job_id[:2] / job.job_id
    paths = WorkspacePaths(workspace=str(job_ws), scratch=str(SCRATCH_ROOT))
    paths.create_directories()
    media_path, aliases = make_local_alias(job)
    cache_local, cache_durable = copy_diar_cache_in(job, paths, config, media_path.stem)
    wavs_previos = set(paths.audio_tmp.glob('*.wav'))
    append_event('processing', job_id=job.job_id,
                 source=source_payload(job.source, job.signature), profile=job.profile)
    es_ultimo = (intentos_previos + 1) >= MAX_INTENTOS_POR_ARCHIVO
    valvula_calidad = bool(PUBLICAR_DEGRADADO_TRAS_AGOTAR) and (
        es_ultimo or bool(PUBLICAR_SI_CALIDAD_NO_MEJORA))
    try:
        marcar_etapa(job.source.relative.as_posix(), 'transcribiendo')
        metadata = process_one(media_path, paths, model, config, diar_engine=diar_engine)
        marcar_etapa(job.source.relative.as_posix(), 'publicando')

        degradado = metadata.get('status') == 'ok_degraded'
        detalle = re.sub(r'^\w+Error:\s*', '', str(metadata.get('diarization_error') or ''))
        if degradado and 'Could not load pyannote pipeline' in str(
                metadata.get('diarization_error') or ''):
            raise FatalDiarizationSetup(detalle)
        criticos = [str(f) for f in (metadata.get('quality_flags') or [])
                    if str(f).startswith('[CRITICAL]')]

        estado, motivo = 'ok', None
        if degradado:
            if not (es_ultimo and PUBLICAR_DEGRADADO_TRAS_AGOTAR):
                raise DiarizationFileError(detalle or 'diarización degradada')
            estado, motivo = 'degradado_sin_diarizacion', (detalle or 'diarización degradada')[:300]
        elif RECHAZAR_CALIDAD_CRITICA and criticos:
            if not valvula_calidad:
                raise QualityRejectedError('Calidad crítica: ' + '; '.join(criticos))
            estado, motivo = 'publicado_con_flags_criticos', ('; '.join(criticos))[:300]

        master_meta = guardar_json_maestro(metadata, job.source, job.signature,
                                           job.profile.get('id', ''))
        try:
            pub = publicar_desde_metadata(job.source, job.output_relative, metadata,
                                          estado, motivo, job.job_id, job.signature,
                                          valvula=valvula_calidad)
        except ValueError as exc:
            raise QualityRejectedError(str(exc)) from exc

        if not source_still_unchanged(job.source):
            raise SourceChangingError('la fuente o su prompt cambió durante la transcripción')

        rec = commit_publicacion(job, pub, 'completed', presentacion_para(job.source.source_id),
                                 master_meta)
        return {'status': 'completed', 'record': rec,
                'quality': pub['estado'], 'motivo': pub['motivo'],
                'relative_path': job.source.relative.as_posix(),
                'output': rec['output']['relative_path'],
                'output_sha256': rec['output']['sha256'],
                'n_extras': len(pub['extras']),
                'words': metadata.get('total_words', 0),
                'duration_seconds': metadata.get('duration_seconds'),
                'speakers': len(metadata.get('speakers_summary') or {}),
                'rtf': metadata.get('real_time_factor')}
    except SourceChangingError:
        raise
    except Exception:
        _rescatar_logs_fallo(paths, job)
        raise
    finally:
        marcar_etapa(None, 'entre archivos')
        with contextlib.suppress(Exception):
            copy_diar_cache_out(cache_local, cache_durable)
        for wav in set(paths.audio_tmp.glob('*.wav')) - wavs_previos:
            with contextlib.suppress(OSError):
                wav.unlink()
        for a in aliases:
            with contextlib.suppress(OSError):
                a.unlink()
        gc.collect()
        with contextlib.suppress(Exception):
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        if job_ws.exists() and is_within(job_ws, JOBS_ROOT) \
                and job_ws.resolve() != JOBS_ROOT.resolve():
            shutil.rmtree(job_ws, ignore_errors=True)


print('⚙️ Motor v5 cargado: perfiles motor/presentación, preflights, planificador '
      'incremental con republicación CPU, dedup por contenido y process_job con válvula.')

## 🟡 7 · Smoke test aislado (opcional, ~2-3 min)

Procesa el **primer archivo** de la entrada con el modelo `small` hacia un workspace descartable en el disco local: verifica GPU, token, ffmpeg y el flujo completo **sin tocar tus salidas ni tu estado**. Recomendado tras cambiar de entorno, de versión de la librería o de cuenta de Colab. `EJECUTAR_SMOKE = False` lo omite (default para `Ejecutar todo`).

In [ ]:
# 🟡 CELDA 7 — SMOKE TEST aislado (no toca salidas ni estado; modelo small)
EJECUTAR_SMOKE = False  # ⇦ True tras cambiar entorno, librería o versión del wheel

if not EJECUTAR_SMOKE:
    print('⏩ Smoke omitido (EJECUTAR_SMOKE=False). Actívalo tras cambiar entorno o librería.')
else:
    import time as _t2

    from speakerscribe import WorkspacePaths, loaded_whisper, process_one

    preflight_storage()
    fuentes_smoke, _avisos, _col = discover_sources()
    if not fuentes_smoke:
        raise RuntimeError(f'No hay archivos procesables en {INPUT_ROOT}')
    objetivo = fuentes_smoke[0]
    estado_probe, _dur, _err = ffprobe_audio(objetivo.path)
    if estado_probe != 'ok':
        raise RuntimeError(f'El primer archivo no es legible ({_err}); '
                           'mueve/repara ese archivo o corre el lote directamente.')

    smoke_cfg = make_engine_config().model_copy(update={
        'model': 'small', 'beam_size': 1, 'batch_size': 8})
    verificar_acceso_hf(smoke_cfg)
    smoke_ws = SCRATCH_ROOT / '_smoke'
    if smoke_ws.exists():
        shutil.rmtree(smoke_ws, ignore_errors=True)
    smoke_paths = WorkspacePaths(workspace=str(smoke_ws), scratch=str(SCRATCH_ROOT))
    smoke_paths.create_directories()

    print(f'🧪 Smoke sobre: {objetivo.relative.as_posix()} ({objetivo.size_bytes / 1e6:.1f} MB)')
    t0 = _t2.time()
    try:
        with loaded_whisper(smoke_cfg) as _smoke_model:
            _smoke = process_one(objetivo.path, smoke_paths, _smoke_model, smoke_cfg)
    finally:
        liberar_memoria_host()

    print(f'✅ Smoke OK en {_t2.time() - t0:.1f} s')
    print(f'   Estado   : {_smoke.get("status")}')
    print(f'   Palabras : {_smoke.get("total_words", 0):,} · idioma: '
          f'{_smoke.get("language_detected", "-")} · hablantes: '
          f'{len(_smoke.get("speakers_summary") or {})}')
    print(f'   Tiempos  : {_smoke.get("timings", {})}')
    _segs = _smoke.get('segments') or []
    if _segs:
        print('\n— Muestra (primeros 3 segmentos) —')
        for _s in _segs[:3]:
            print(f'  [{format_ts(_s.get("start", 0))}] '
                  f'{_s.get("speaker") or "?"}: {str(_s.get("text") or "").strip()[:90]}')
    shutil.rmtree(smoke_ws, ignore_errors=True)
    print('\n🧹 Workspace del smoke eliminado. El estado real quedó intacto.')

## 🟢 8 · 🚀 Ejecutar el lote (incremental, predictivo, con apagado garantizado)

Whisper y pyannote se cargan **una sola vez** (y Community-1 se valida **antes** que Whisper). Por cada archivo: `verificar → transcribir+diarizar → reparar hablantes → publicar entregables → JSON maestro → commit → _resumen.md`. Todo confirmado **antes** de pasar al siguiente.

Protecciones activas: guardia de RAM (recicla modelos / cierre limpio pre-OOM), presupuesto de sesión con RTF aprendido, *circuit breaker* de diarización (2 fallos seguidos cortan el lote), válvula anti-pérdida y apagado en `finally` — cancelable con ⏹ durante la cuenta regresiva.

In [ ]:
%%time
# 🟢 CELDA 8 — 🚀 EJECUCIÓN: incremental, predictiva y con apagado garantizado en finally
import math as _math
import time as _time

from tqdm.auto import tqdm

if 'process_job' not in globals() or 'create_planner_state' not in globals():
    raise RuntimeError('Ejecuta antes las CELDAS 3 a 6.')


def _rtf_aprendido(events: list[dict[str, Any]]) -> float:
    """Percentil 10 del RTF histórico de ESTE modelo — ya es conservador.

    Sin ≥5 métricas propias se usa el piso RTF_ASR_CONSERVADOR. El p10 de las
    velocidades observadas ya es la cola lenta: no se recorta otra vez encima
    (ese doble recorte aplazaba audios que sí cabían).
    """
    vistos = sorted(float(r.get('real_time_factor'))
                    for r in events if r.get('event') == 'job_metrics'
                    and (r.get('profile') or {}).get('asr_model') == MODELO
                    and isinstance(r.get('real_time_factor'), (int, float))
                    and 0 < float(r.get('real_time_factor')) < 100)[-100:]
    if len(vistos) < 5:
        return float(RTF_ASR_CONSERVADOR)
    p10 = vistos[max(0, int(_math.floor(0.1 * (len(vistos) - 1))))]
    return max(1.0, min(60.0, p10))


def _escribir_revision(valvula: list[dict], errores: list[dict], bloqueados: int) -> None:
    with contextlib.suppress(Exception):
        atomic_write_json(REVIEW_PATH, {
            'generado_utc': utc_now(), 'run_id': RUN_ID,
            'publicados_marcados': valvula, 'fallidos_esta_sesion': errores,
            'bloqueados_por_intentos': bloqueados,
            'nota': 'Los marcados SÍ tienen .txt (revisa el encabezado "estado:"). '
                    'Los bloqueados se liberan con FORZAR_REPROCESO o al cambiar el archivo.'})


def escribir_resumen_md(resumen: dict[str, Any], plan_tot: dict[str, Any],
                        filas: list[dict[str, Any]]) -> None:
    """Regenera `_resumen.md` en la salida (legible, tras CADA archivo)."""
    with contextlib.suppress(Exception):
        L = [f'# Resumen SpeakerScribe v5 — {utc_now()}', '',
             f'- **Run:** `{RUN_ID[:12]}` · notebook v{NOTEBOOK_VERSION}',
             f'- **Entrada:** `{INPUT_ROOT}`',
             f'- **Salida:** `{OUTPUT_ROOT}`',
             f'- **Motor:** `{MODELO}` · idioma {IDIOMA or "auto"} · beam {BEAM_SIZE} '
             f'· batch {BATCH_SIZE} · anti-alucinación {ANTI_ALUCINACION}',
             f'- **Sesión:** ✔ {resumen.get("completados_ok", 0)} OK · '
             f'🧯 {resumen.get("completados_degradados", 0)} marcados · '
             f'♻️ {resumen.get("reusados", 0)} reusados · '
             f'📝 {resumen.get("republicados", 0)} republicados (CPU) · '
             f'✖ {resumen.get("fallidos", 0)} fallidos · '
             f'⏭️ {resumen.get("diferidos_por_tiempo", 0)} diferidos',
             f'- **Plan acumulado:** {plan_tot.get("ya_listos", 0)} ya listos · '
             f'{plan_tot.get("invalidos", 0)} inválidos · '
             f'{plan_tot.get("bloqueados", 0)} bloqueados · '
             f'{plan_tot.get("perfil_cambiado", 0)} con otro perfil de motor',
             '', '## Archivos tocados en esta sesión', '',
             '| Archivo | Resultado | Duración | Palabras | Detalle |',
             '|---|---|---|---|---|']
        for f_ in filas[-400:]:
            L.append(f'| `{f_.get("rel")}` | {f_.get("estado")} '
                     f'| {format_ts(f_.get("dur_s") or 0)} '
                     f'| {f_.get("palabras") or 0:,} | {str(f_.get("detalle") or "—")[:90]} |')
        L += ['', '_Estado completo: `.speakerscribe_state/` · '
                  'revisión: `pendientes_revision.json` · caja negra: `telemetria.jsonl`_']
        atomic_write_text(RESUMEN_MD_PATH, '\n'.join(L) + '\n')


def run_recursive_batch() -> dict[str, Any]:
    global ACTIVE_LOCK, MOTIVO_FIN, RESUMEN
    import speakerscribe
    from speakerscribe import DiarizationEngine, loaded_whisper

    MOTIVO_FIN = None
    run_started = float(globals().get('SESION_INICIADA_MONOTONIC', _time.monotonic()))
    usable = (float(MAX_MINUTOS_SESION) - float(MARGEN_CIERRE_MINUTOS)) \
        if MAX_MINUTOS_SESION > 0 else float('inf')

    def minutos() -> float:
        return (_time.monotonic() - run_started) / 60

    def sin_presupuesto() -> bool:
        return minutos() >= usable

    resumen: dict[str, Any] = {'run_id': RUN_ID, 'inicio_utc': utc_now(),
                               'completados_ok': 0, 'completados_degradados': 0,
                               'reusados': 0, 'republicados': 0, 'fallidos': 0,
                               'diferidos_por_tiempo': 0, 'audio_s': 0.0}
    valvula_pub: list[dict] = []
    errores_ses: list[dict] = []
    filas_md: list[dict] = []
    lock = StateLock(LOCK_PATH)
    excepcion_fatal = False
    plan_tot: dict[str, Any] = {}
    vigilante = None
    if VIGILANCIA_RECURSOS:
        vigilante = VigilanteRecursos(TELEMETRY_PATH, 60, int(LATIDO_MINUTOS))
        vigilante.start()
    try:
        marcar_etapa(None, 'preflight')
        preflight_storage()
        lock.acquire()
        ACTIVE_LOCK = lock
        identidad = bind_workspace_identity()
        config = make_engine_config()
        recuperados = recover_prepared(read_events())
        fuentes, avisos, colisiones = discover_sources()
        for a in avisos:
            print('Advertencia:', a)
        eventos0 = read_events()
        estado_plan = create_planner_state(fuentes, colisiones, eventos0)
        ordenadas = estado_plan['ordered']
        eff_rtf = _rtf_aprendido(eventos0)
        plan_tot.update({'detectados': len(fuentes), 'examinados': 0, 'ya_listos': 0,
                         'republicados': 0, 'reusados': 0, 'invalidos': 0,
                         'pospuestos': 0, 'bloqueados': 0, 'probe_reintentable': 0,
                         'perfil_cambiado': 0, 'sin_maestro': 0,
                         'diferidos_presupuesto': 0})
        resumen.update(recuperados_commit=recuperados, identidad=identidad,
                       rtf_estimacion=round(eff_rtf, 2),
                       auditorias={'n': estado_plan['audit_n'],
                                   'gb': estado_plan['audit_gb']},
                       entorno={'nota': 'GPU sin cargar: aún no hubo trabajo',
                                'versiones': {'speakerscribe': speakerscribe.__version__}})
        append_event('batch_started', settings={
            'notebook_version': NOTEBOOK_VERSION,
            'input_root_sha256': canonical_hash(str(INPUT_ROOT)),
            'model': MODELO, 'language': IDIOMA or None,
            'beam': int(BEAM_SIZE), 'batch': int(BATCH_SIZE),
            'presentacion_id': PRESENTACION_GLOBAL['id'],
            'detectados': len(fuentes)})

        print('\n════════ INICIO INCREMENTAL ════════')
        print(f'  Detectados: {len(fuentes)}  ·  primero nuevos/cambiados, '
              f'luego {estado_plan["audit_n"]} auditorías (≤{estado_plan["audit_gb"]} GB)')
        print('  Presupuesto: '
              + (f'{MAX_MINUTOS_SESION} min (margen {MARGEN_CIERRE_MINUTOS}); '
                 f'ya consumidos {minutos():.1f}' if MAX_MINUTOS_SESION else 'sin límite')
              + f'  ·  RTF para estimar: {eff_rtf:.2f}x')

        modelo = None
        motor_diar = None
        procesados = 0
        reciclajes = 0
        desde_reciclaje = 0
        ram_pico = ram_pct_actual()
        consecutivos_diar = 0
        cursor = 0
        barra = tqdm(total=len(ordenadas), unit='fuente', desc='Verificar/Procesar')

        def _fila(rel: str, estado: str, dur_s: float = 0.0, palabras: int = 0,
                  detalle: str | None = None) -> None:
            filas_md.append({'rel': rel, 'estado': estado, 'dur_s': dur_s,
                             'palabras': palabras, 'detalle': detalle})
            escribir_resumen_md(resumen, plan_tot, filas_md)

        with contextlib.ExitStack() as pila:
            pila_modelos = pila.enter_context(contextlib.ExitStack())

            def reciclar_modelos(motivo: str) -> None:
                """Libera Whisper y pyannote, purga el heap y deja que se recarguen.

                La fuga medida (~71 MB de RAM por archivo, VRAM plana) vive en
                el proceso Python: soltar los modelos es la única purga profunda
                sin reiniciar el runtime. Cuesta ~35 s de recarga y deja
                constancia del antes/después en el journal.
                """
                nonlocal modelo, motor_diar, reciclajes, desde_reciclaje
                antes = ram_pct_actual()
                pila_modelos.close()
                modelo = None
                motor_diar = None
                liberar_memoria_host()
                despues = ram_pct_actual()
                reciclajes += 1
                desde_reciclaje = 0
                append_event('models_recycled', reason=motivo, ram_pct_before=antes,
                             ram_pct_after=despues, files_done=procesados)
                tqdm.write(f'   🔄 Modelos reciclados ({motivo}): RAM {antes:.0f}% → '
                           f'{despues:.0f}%  ·  se recargan en el próximo archivo')

            while cursor < len(ordenadas) and MOTIVO_FIN is None:
                if sin_presupuesto():
                    MOTIVO_FIN = 'limite_sesion'
                    break
                if MAX_ARCHIVOS_POR_SESION > 0 and procesados >= MAX_ARCHIVOS_POR_SESION:
                    MOTIVO_FIN = 'limite_archivos'
                    break
                bloque = ordenadas[cursor:cursor + int(TAMANO_BLOQUE_PLANIFICACION)]
                jobs, conteo = plan_block(config, speakerscribe.__version__,
                                          estado_plan, bloque, sin_presupuesto)
                for k, v in conteo.items():
                    plan_tot[k] = plan_tot.get(k, 0) + v
                resumen['republicados'] += conteo.get('republicados', 0)
                resumen['reusados'] += conteo.get('reusados', 0)
                cursor += conteo['examinados']
                barra.update(conteo['examinados'])
                for fte in bloque[:conteo['examinados']]:
                    if fte.source_id in estado_plan['strict_ids']:
                        lock.assert_owned()
                        atomic_write_json(AUDIT_CURSOR_PATH,
                                          {'last_source_id': fte.source_id,
                                           'relative_path': fte.relative.as_posix(),
                                           'updated_utc': utc_now()})
                if conteo.get('republicados') or conteo.get('reusados'):
                    escribir_resumen_md(resumen, plan_tot, filas_md)
                if conteo['examinados'] == 0 and conteo['diferidos_presupuesto']:
                    MOTIVO_FIN = 'limite_sesion'
                    break

                for job in jobs:
                    if sin_presupuesto():
                        MOTIVO_FIN = 'limite_sesion'
                        break
                    # ── Guardia de RAM: antes de comprometer otro archivo ──
                    _ram = ram_pct_actual()
                    ram_pico = max(ram_pico, _ram)
                    if _ram >= float(RAM_PCT_DETENER):
                        MOTIVO_FIN = 'limite_ram'
                        tqdm.write(f'   🛡️ RAM al {_ram:.0f}% (tope {RAM_PCT_DETENER}%): '
                                   'cierre LIMPIO antes del OOM.')
                        break
                    _toca_n = (int(RECICLAR_CADA_N_ARCHIVOS) > 0
                               and desde_reciclaje >= int(RECICLAR_CADA_N_ARCHIVOS))
                    if modelo is not None and (_ram >= float(RAM_PCT_RECICLAR) or _toca_n):
                        reciclar_modelos(f'ram {_ram:.0f}%' if not _toca_n
                                         else f'cada {RECICLAR_CADA_N_ARCHIVOS}')
                        if ram_pct_actual() >= float(RAM_PCT_DETENER):
                            MOTIVO_FIN = 'limite_ram'
                            tqdm.write('   🛡️ El reciclaje no bastó: cierre limpio.')
                            break
                    if MAX_ARCHIVOS_POR_SESION > 0 and procesados >= MAX_ARCHIVOS_POR_SESION:
                        MOTIVO_FIN = 'limite_archivos'
                        break
                    rel = job.source.relative.as_posix()
                    marcar_etapa(rel, 'planificado')
                    tqdm.write(f'▶ {rel}  ({job.source.size_bytes / 1e6:.1f} MB'
                               + (f', {format_ts(job.duration_seconds)}'
                                  if job.duration_seconds else '') + ')')

                    dur_min = (max(0.0, job.duration_seconds or 0.0) / 60) or 60.0
                    est = dur_min / max(eff_rtf, 1e-6) \
                        + dur_min / float(RTF_DIAR_CONSERVADOR) + int(SOBRECOSTE_ARCHIVO_MIN)
                    restante = usable - minutos()
                    if MAX_MINUTOS_SESION > 0 and est > restante:
                        resumen['diferidos_por_tiempo'] += 1
                        tqdm.write(f'   ⏭️ APLAZADO por tiempo: ~{est:.0f} min estimados, '
                                   f'quedan {restante:.0f}. Se retoma en la próxima sesión.')
                        continue

                    if modelo is None:
                        marcar_etapa(rel, 'cargando modelos')
                        resumen['entorno'] = preflight_gpu(config, [job])
                        tqdm.write(f'   🖥️ {resumen["entorno"]["gpu"]} '
                                   f'({resumen["entorno"]["vram_gb"]:.1f} GB) · '
                                   'validando Community-1 ANTES de cargar Whisper…')
                        motor_diar = pila_modelos.enter_context(DiarizationEngine(config))
                        try:
                            if hasattr(motor_diar, '_ensure_loaded'):
                                motor_diar._ensure_loaded()
                        except Exception as exc:
                            append_event('diarization_setup_failed',
                                         error=sanitize_error(exc))
                            MOTIVO_FIN = 'abort_fatal'
                            tqdm.write('🛑 ABORT FATAL: la diarización no puede '
                                       'inicializar (token/términos/CUDA). Se detiene '
                                       'el lote sin gastar intentos.')
                            tqdm.write('   Detalle: ' + sanitize_error(exc)[:220])
                            tqdm.write('   → https://huggingface.co/pyannote/'
                                       'speaker-diarization-community-1')
                            break
                        modelo = pila_modelos.enter_context(loaded_whisper(config))

                    procesados += 1
                    desde_reciclaje += 1
                    try:
                        r = process_job(job, modelo, motor_diar, config,
                                        estado_plan['indices']['intentos'].get(job.job_id, 0))
                        consecutivos_diar = 0
                        rec = r.get('record')
                        if rec is not None:
                            estado_plan['indices']['ok_por_fuente'].setdefault(
                                job.source.source_id, []).append(rec)
                            estado_plan['indices']['sucias'].discard(job.source.source_id)
                            if job.signature and not job.signature.startswith('fast:'):
                                estado_plan['indices']['ok_por_contenido_perfil'].setdefault(
                                    (job.signature, job.profile_id), []).append(rec)
                        resumen['audio_s'] += float(r.get('duration_seconds') or 0)
                        append_event('job_metrics',
                                     source=source_payload(job.source, job.signature),
                                     profile=job.profile,
                                     duration_minutes=(r.get('duration_seconds') or 0) / 60,
                                     real_time_factor=r.get('rtf'))
                        if r['quality'] in ('ok', 'ok_sin_voz'):
                            resumen['completados_ok'] += 1
                            _fila(rel, '✔ ' + r['quality'],
                                  float(r.get('duration_seconds') or 0),
                                  int(r.get('words') or 0), r.get('output'))
                            tqdm.write(f'   ✔ OK → {r["output"]} · {r["words"]:,} palabras · '
                                       f'{r["speakers"]} hablantes · '
                                       f'{r["n_extras"]} extra(s)'
                                       + (f' · RTF {r["rtf"]:.1f}x' if r.get('rtf') else ''))
                        else:
                            resumen['completados_degradados'] += 1
                            valvula_pub.append({'archivo': rel, 'salida': r['output'],
                                                'estado': r['quality'],
                                                'motivo': r.get('motivo')})
                            _fila(rel, '🧯 ' + r['quality'],
                                  float(r.get('duration_seconds') or 0),
                                  int(r.get('words') or 0), r.get('motivo'))
                            tqdm.write(f'   🧯 PUBLICADO MARCADO ({r["quality"]}) → {r["output"]}')
                    except FatalDiarizationSetup as exc:
                        append_event('failed_retryable', job_id=job.job_id,
                                     source=source_payload(job.source, job.signature),
                                     profile=job.profile, error=sanitize_error(exc),
                                     stage='diar_setup')
                        resumen['fallidos'] += 1
                        MOTIVO_FIN = 'abort_fatal'
                        tqdm.write('🛑 ABORT FATAL: la diarización no puede inicializar '
                                   '(token/términos/CUDA). Se detiene el lote.')
                        tqdm.write('   → https://huggingface.co/pyannote/'
                                   'speaker-diarization-community-1')
                        break
                    except DiarizationFileError as exc:
                        consecutivos_diar += 1
                        append_event('failed_retryable', job_id=job.job_id,
                                     source=source_payload(job.source, job.signature),
                                     profile=job.profile, error=sanitize_error(exc),
                                     stage='diar_file')
                        resumen['fallidos'] += 1
                        errores_ses.append({'archivo': rel, 'error': sanitize_error(exc)[:200]})
                        _fila(rel, '✖ diarización', detalle=sanitize_error(exc)[:90])
                        tqdm.write('   ✖ DIARIZACIÓN de este archivo falló (reintentable): '
                                   + sanitize_error(exc)[:160])
                        if consecutivos_diar >= 2:
                            append_event('diarization_setup_failed',
                                         error='dos archivos consecutivos sin diarizar; '
                                               'se presume fallo global')
                            MOTIVO_FIN = 'abort_fatal'
                            tqdm.write('🛑 Dos archivos consecutivos sin diarizar: se corta '
                                       'el lote para proteger tu cuota.')
                            break
                    except SourceChangingError as exc:
                        append_event('source_changed', job_id=job.job_id,
                                     source=source_payload(job.source, job.signature),
                                     profile=job.profile, error=sanitize_error(exc))
                        _fila(rel, '⏸ pospuesto', detalle='el archivo cambió')
                        tqdm.write('   ⏸ POSPUESTO (el archivo cambió).')
                    except QualityRejectedError as exc:
                        append_event('quality_rejected', job_id=job.job_id,
                                     source=source_payload(job.source, job.signature),
                                     profile=job.profile, error=sanitize_error(exc))
                        resumen['fallidos'] += 1
                        errores_ses.append({'archivo': rel, 'error': sanitize_error(exc)[:200]})
                        _fila(rel, '✖ calidad', detalle=sanitize_error(exc)[:90])
                        tqdm.write('   ✖ RECHAZADO por calidad (reintentará): '
                                   + sanitize_error(exc)[:160])
                    except KeyboardInterrupt:
                        MOTIVO_FIN = 'interrumpido'
                        tqdm.write('⏹ Interrumpido: lo confirmado está a salvo.')
                        break
                    except Exception as exc:
                        tb = re.sub(r'hf_[A-Za-z0-9_-]{8,}', 'hf_***redactado***',
                                    ''.join(traceback.format_exception(exc))[-8000:])
                        append_event('failed_retryable', job_id=job.job_id,
                                     source=source_payload(job.source, job.signature),
                                     profile=job.profile, error=sanitize_error(exc),
                                     traceback=tb)
                        resumen['fallidos'] += 1
                        errores_ses.append({'archivo': rel, 'error': sanitize_error(exc)[:200]})
                        _fila(rel, '✖ error', detalle=sanitize_error(exc)[:90])
                        tqdm.write('   ✖ ERROR: ' + sanitize_error(exc)[:180])
                    finally:
                        liberar_memoria_host()
                        barra.set_postfix(ram=f'{ram_pct_actual():.0f}%',
                                          ok=resumen['completados_ok'],
                                          deg=resumen['completados_degradados'],
                                          rep=resumen['republicados'],
                                          reuso=resumen['reusados'],
                                          err=resumen['fallidos'])
        barra.close()
        if MOTIVO_FIN is None:
            MOTIVO_FIN = 'completo'
    except KeyboardInterrupt:
        MOTIVO_FIN = 'interrumpido'
        print('⏹ Interrumpido por el usuario: lo confirmado está a salvo en Drive.')
    except Exception as exc:
        excepcion_fatal = True
        MOTIVO_FIN = MOTIVO_FIN or 'error_orquestador'
        resumen['error_fatal'] = sanitize_error(exc)
        print('🛑 ERROR FATAL del orquestador:', sanitize_error(exc))
        print(re.sub(r'hf_[A-Za-z0-9_-]{8,}', 'hf_***',
                     ''.join(traceback.format_exception(exc))[-3000:]))
        with contextlib.suppress(Exception):
            append_event('batch_aborted', error=sanitize_error(exc))
    finally:
        resumen.update(fin_utc=utc_now(), motivo_fin=MOTIVO_FIN,
                       minutos_sesion=round(
                           (_time.monotonic() - run_started) / 60, 1),
                       plan=dict(plan_tot), ram_pct_final=ram_pct_actual(),
                       reciclajes=int(locals().get('reciclajes', 0)))
        with contextlib.suppress(Exception):
            _escribir_revision(valvula_pub, errores_ses, plan_tot.get('bloqueados', 0))
        with contextlib.suppress(Exception):
            atomic_write_json(SUMMARY_PATH, resumen)
        with contextlib.suppress(Exception):
            escribir_resumen_md(resumen, plan_tot, filas_md)
        with contextlib.suppress(Exception):
            append_event('batch_finished', summary={
                k: v for k, v in resumen.items() if k not in ('plan', 'entorno')})
        lock.release()
        ACTIVE_LOCK = None
        RESUMEN = resumen
        if vigilante is not None:
            vigilante.detener(f'fin:{MOTIVO_FIN}')
            resumen['pico_ram_gb'] = vigilante.pico_ram_gb
            resumen['ultima_telemetria'] = vigilante.ultima

        _min_ses = float(resumen['minutos_sesion'])
        entregables = (resumen['completados_ok'] + resumen['completados_degradados']
                       + resumen['reusados'] + resumen['republicados'])
        sospechoso = (bool(APAGAR_SOLO_SI_HUBO_TRABAJO) and MOTIVO_FIN == 'completo'
                      and entregables == 0 and _min_ses < 10
                      and plan_tot.get('ya_listos', 0) == 0)
        # 'limite_ram' NUNCA apaga: la fuga vive en el proceso; lo que hace falta
        # es REINICIAR EL ENTORNO, no soltar la VM. 'abort_fatal' tampoco: se
        # necesita la máquina viva para leer el error.
        apagar = bool(APAGAR_AL_FINAL) and not sospechoso and (
            MOTIVO_FIN in ('completo', 'limite_sesion', 'limite_archivos')
            or (excepcion_fatal and bool(APAGAR_TRAS_ERROR_FATAL)))
        print('\n══════════ RESUMEN DE LA SESIÓN ══════════')
        print(f'  ✔ OK: {resumen["completados_ok"]}   🧯 Marcados: '
              f'{resumen["completados_degradados"]}   ♻️ Reusados: {resumen["reusados"]}'
              f'   📝 Republicados (CPU): {resumen["republicados"]}'
              f'   ✖ Fallidos: {resumen["fallidos"]}   ⏭️ Diferidos: '
              f'{resumen["diferidos_por_tiempo"]}')
        print(f'  🎧 Audio confirmado: {format_ts(resumen["audio_s"])}'
              f'   ·   motivo fin: {MOTIVO_FIN}')
        print(f'  📁 {OUTPUT_ROOT}\n  📊 {RESUMEN_MD_PATH.name}  ·  🧾 {STATE_ROOT.name}/ '
              f'(revisión: pendientes_revision.json)')
        if plan_tot.get('sin_maestro'):
            print(f'  ⚠️ {plan_tot["sin_maestro"]} archivo(s) con presentación desactualizada '
                  'SIN JSON maestro: se conservó la salida previa. Para regenerarlos '
                  'con el formato nuevo: FORZAR_REPROCESO=True (GPU).')
        if MOTIVO_FIN == 'limite_ram':
            print('\n🛡️  DETENIDO POR RAM — no es un fallo: es la guardia haciendo su trabajo.')
            print('   Todo lo confirmado está en Drive. Para continuar donde quedó:')
            print('   Entorno de ejecución → Reiniciar sesión → Ejecutar todo.')
            print(f'   (reciclajes de modelo en esta sesión: {resumen.get("reciclajes", 0)})')
        if resumen.get('pico_ram_gb'):
            _ult = resumen.get('ultima_telemetria') or {}
            print(f'  🩺 Pico de RAM: {resumen["pico_ram_gb"]:.1f} GB de '
                  f'{_ult.get("ram_total_gb", "?")} GB  ·  telemetría: telemetria.jsonl')
        if sospechoso:
            print('\n⚠️  El lote terminó en <10 min SIN entregables ni confirmados previos.')
            print('   Eso no es un éxito: revisa pendientes_revision.json y events.jsonl.')
            print('   Máquina VIVA a propósito para que puedas diagnosticar.')
        if apagar and IN_COLAB:
            print(f'\n🛑 Apagado en {RETRASO_APAGADO_S} s (motivo: {MOTIVO_FIN}) '
                  '— pulsa ⏹ para cancelar.')
            try:
                restante = int(RETRASO_APAGADO_S)
                while restante > 0:
                    print(f'   … {restante:>3d} s', flush=True)
                    _time.sleep(min(5, restante))
                    restante -= 5
            except KeyboardInterrupt:
                print('✋ Apagado CANCELADO.')
            else:
                print('💾 flush_and_unmount…')
                with contextlib.suppress(Exception):
                    from google.colab import drive as _gd
                    _gd.flush_and_unmount()
                print('🔌 runtime.unassign()')
                from google.colab import runtime as _gr
                _gr.unassign()
        elif apagar:
            print('\nℹ️ (Fuera de Colab: no hay VM que apagar.)')
        else:
            print('\n🔌 Sesión activa (motivo:', str(MOTIVO_FIN) + ').',
                  'CELDA 12 para apagar manualmente.')
    return resumen


RESUMEN = run_recursive_batch()

## 🟢 9 · Estado global del lote (sin GPU)

Censo completo: cruza los archivos encontrados con el journal y dice qué está confirmado, qué falta, qué quedó marcado y qué está bloqueado. Ejecutable en cualquier sesión (solo necesita Drive + CELDAS 3-6). Verificación por `stat` (rápida); la validación fuerte por hash la hace la auditoría rotatoria del lote.

In [ ]:
# 🟢 CELDA 9 — ESTADO GLOBAL del lote (solo lectura; sin GPU, sin hashes)
if 'discover_sources' not in globals():
    raise RuntimeError('Ejecuta antes las CELDAS 3 a 6.')

preflight_storage()
_fuentes, _avisos, _cols = discover_sources()
_indices = build_indices(read_events())

_conf = _marcados = _pendientes = _bloq = _inval = 0
_pend_detalle: list[str] = []
_marc_detalle: list[tuple[str, str]] = []
for _s in _fuentes:
    _prior = latest_success_record(_indices['ok_por_fuente'].get(_s.source_id, []))
    if _prior and source_stat_matches(_prior, _s) and output_record_stat_is_valid(_prior):
        _q = str(_prior.get('quality') or 'ok')
        if _q in ('ok', 'ok_sin_voz'):
            _conf += 1
        else:
            _marcados += 1
            _marc_detalle.append((_s.relative.as_posix(), _q))
        continue
    _inv = _indices['invalidos'].get(_s.source_id)
    if _inv and source_stat_matches(_inv, _s):
        _inval += 1
        continue
    _pendientes += 1
    if len(_pend_detalle) < 15:
        _pend_detalle.append(_s.relative.as_posix())

for _n_int in _indices['intentos'].values():
    if _n_int >= MAX_INTENTOS_POR_ARCHIVO:
        _bloq += 1

print('═' * 68)
print(f'  ESTADO GLOBAL — {len(_fuentes)} archivo(s) en la entrada')
print('═' * 68)
print(f'  ✔ Confirmados OK            : {_conf:>5d}')
print(f'  🧯 Confirmados MARCADOS      : {_marcados:>5d}   (revisa pendientes_revision.json)')
print(f'  ⏳ Pendientes / cambiados    : {_pendientes:>5d}')
print(f'  🚫 Inválidos (sin audio)     : {_inval:>5d}')
print(f'  ⛔ Jobs con intentos agotados: {_bloq:>5d}   (FORZAR_REPROCESO los libera)')
if _pend_detalle:
    print('\n  Próximos pendientes:')
    for _p in _pend_detalle:
        print(f'     · {_p}')
    if _pendientes > len(_pend_detalle):
        print(f'     … y {_pendientes - len(_pend_detalle)} más')
if _marc_detalle:
    print('\n  Marcados (publicados con reservas):')
    for _rel, _q in _marc_detalle[:10]:
        print(f'     🧯 {_rel}  ({_q})')
    if len(_marc_detalle) > 10:
        print(f'     … y {len(_marc_detalle) - 10} más')
print('─' * 68)
print(f'  📁 Salida     : {OUTPUT_ROOT}')
print(f'  📊 Resumen    : {RESUMEN_MD_PATH}')
print(f'  🧾 Journal    : {EVENTS_PATH.name} · revisión: {REVIEW_PATH.name}')
if _pendientes:
    print('\n  ↻ Hay pendientes: «Ejecutar todo» y el lote retoma exactamente donde quedó.')
else:
    print('\n  ✅ Nada pendiente: la colección está completa con la configuración actual.')

## 🟡 10 · Renombrar hablantes (opcional, sin GPU)

Escucha el inicio del audio, identifica quién es `SPEAKER_00`, edita `MAPPING` y pon `RUN_RENAME = True`. El mapping queda **persistente** en el estado: todos los entregables del archivo se republican con los nombres reales, y cualquier republicación futura los conserva. Requiere el JSON maestro (`GUARDAR_JSON_MAESTRO=True` cuando se procesó).

In [ ]:
# 🟡 CELDA 10 — RENOMBRAR HABLANTES y republicar (CPU, sin GPU)
# ✏️ Edita estos tres valores:
ARCHIVO_REL = ''      # ⇦ ruta relativa tal como aparece en la lista de abajo
MAPPING = {
    'SPEAKER_00': 'Entrevistador',
    'SPEAKER_01': 'Invitada',
}
RUN_RENAME = False    # ⇦ True cuando MAPPING esté listo

if 'discover_sources' not in globals():
    raise RuntimeError('Ejecuta antes las CELDAS 3 a 6.')

preflight_storage()
_fuentes_r, _av, _co = discover_sources()
_indices_r = build_indices(read_events())
_hechos = [s for s in _fuentes_r
           if latest_success_record(_indices_r['ok_por_fuente'].get(s.source_id, []))]

if not RUN_RENAME:
    print('📋 Archivos con resultado publicado (usa la ruta EXACTA en ARCHIVO_REL):')
    for _s in _hechos[-25:]:
        _m = cargar_renombres(_s.source_id)
        print(f'   · {_s.relative.as_posix()}' + (f'   (renombres: {_m})' if _m else ''))
    if not _hechos:
        print('   (aún no hay archivos publicados — corre primero la CELDA 8)')
    print('\n⏩ Edita ARCHIVO_REL y MAPPING, y pon RUN_RENAME=True para aplicar.')
else:
    _obj = next((s for s in _hechos if s.relative.as_posix() == ARCHIVO_REL), None)
    if _obj is None:
        raise ValueError(f'No encuentro {ARCHIVO_REL!r} entre los archivos publicados. '
                         'Copia la ruta exacta de la lista (RUN_RENAME=False la imprime).')
    _prior = latest_success_record(_indices_r['ok_por_fuente'].get(_obj.source_id, []))
    if leer_json_maestro(_obj.source_id) is None:
        raise RuntimeError('Este archivo no tiene JSON maestro (¿GUARDAR_JSON_MAESTRO estaba '
                           'en False al procesarlo?). Reprocésalo con FORZAR_REPROCESO=True.')
    _sha = guardar_renombres(_obj.source_id, MAPPING)
    append_event('speakers_renamed', source=source_payload(_obj),
                 rename_sha=_sha, mapping=MAPPING)
    _lock_r = StateLock(LOCK_PATH)
    _lock_r.acquire()
    ACTIVE_LOCK = _lock_r
    try:
        _rec = republicar_fuente(_obj, _prior)
    finally:
        _lock_r.release()
        ACTIVE_LOCK = None
    if _rec is None:
        raise RuntimeError('No se pudo republicar (falta el JSON maestro).')
    print(f'✅ Renombrado y republicado: {_rec["output"]["relative_path"]}')
    print(f'   Mapping persistente: {MAPPING}')
    print(f'   Extras regenerados : {len(_rec.get("extras") or [])}')
    print('   Las republicaciones futuras conservarán estos nombres.')

## 🟢 11 · 🔬 Autopsia — ejecutar cuando una sesión muera sin terminar

Ejecútala en una sesión NUEVA, después de montar Drive (CELDA 1) y la CELDA 3. No necesita GPU, ni token, ni cargar modelos. Dictamina: ¿OOM (RAM contra el techo), corte externo de Colab (todo plano), o cierre ordenado? Y muestra qué está frenando el rendimiento.

In [ ]:
# 🟢 CELDA 11 — AUTOPSIA: ¿por qué murió la sesión anterior?
import json as _json_a
from collections import Counter as _Counter
from pathlib import Path as _Path_a

_ESTADO_A = _Path_a(RUTA_SALIDA).expanduser().resolve() / '.speakerscribe_state'
_TELE_A = _ESTADO_A / 'telemetria.jsonl'
_RES_A = _ESTADO_A / 'last_summary.json'
_EV_A = _ESTADO_A / 'events.jsonl'


def _leer_jsonl_a(ruta, ultimas: int = 0) -> list:
    """Lee un JSONL tolerando líneas truncadas por un corte de energía."""
    if not ruta.is_file():
        return []
    filas = []
    with ruta.open(encoding='utf-8', errors='replace') as f:
        for linea in f:
            linea = linea.strip()
            if not linea:
                continue
            try:
                filas.append(_json_a.loads(linea))
            except _json_a.JSONDecodeError:
                continue          # última línea a medio escribir: es normal
    return filas[-ultimas:] if ultimas else filas


print('═' * 70)
print('AUTOPSIA DE LA ÚLTIMA SESIÓN')
print('═' * 70)

# ── 1 · ¿Terminó ordenadamente o desapareció? ─────────────────────────
if _RES_A.is_file():
    _r = _json_a.loads(_RES_A.read_text(encoding='utf-8'))
    print(f"\n🧾 Último resumen escrito: motivo_fin = {_r.get('motivo_fin')} "
          f"· {_r.get('minutos_sesion')} min")
    print(f"   ✔ {_r.get('completados_ok', 0)} OK · 🧯 {_r.get('completados_degradados', 0)} "
          f"marcados · ♻️ {_r.get('reusados', 0)} reusados · 📝 {_r.get('republicados', 0)} "
          f"republicados · ✖ {_r.get('fallidos', 0)} fallidos · "
          f"⏭️ {_r.get('diferidos_por_tiempo', 0)} diferidos")
    if _r.get('error_fatal'):
        print(f"   🛑 error_fatal registrado: {_r['error_fatal']}")
else:
    print('\n🧾 No hay last_summary.json: la sesión NUNCA llegó al bloque final.')
    print('   ⇒ la VM murió sin control (corte externo, OOM o kernel caído).')

# ── 2 · Caja negra: ¿RAM subiendo (OOM) o todo plano (corte externo)? ──
_tele = _leer_jsonl_a(_TELE_A)
if not _tele:
    print('\n🩺 Sin telemetría. Actívala con VIGILANCIA_RECURSOS=True y repite el lote.')
else:
    _ult = _tele[-1]
    _cola = [t for t in _tele if isinstance(t.get('ram_usada_gb'), (int, float))][-15:]
    _pico = max((float(t.get('ram_usada_gb') or 0) for t in _tele), default=0.0)
    _total = _ult.get('ram_total_gb') or 0
    print(f'\n🩺 Telemetría: {len(_tele)} muestras · última a los '
          f"{_ult.get('min_sesion')} min (motivo: {_ult.get('motivo')})")
    print(f"   RAM pico {_pico:.1f} GB de {_total} GB · VRAM última "
          f"{_ult.get('vram_reservada_gb', '?')} GB · disco {_ult.get('disco_libre_gb', '?')} GB")
    print(f"   Estaba en: {_ult.get('archivo') or '—'} · etapa {_ult.get('etapa') or '—'}")
    if len(_cola) >= 5:
        _subida = float(_cola[-1]['ram_usada_gb']) - float(_cola[0]['ram_usada_gb'])
        print(f'   Tendencia de RAM en las últimas {len(_cola)} muestras: {_subida:+.2f} GB')
        if _total and _pico / float(_total) > 0.90:
            print('   ⇒ VEREDICTO: la RAM llegó al tope ⇒ el proceso se quedó sin memoria.')
            print('      Acción: baja RAM_PCT_RECICLAR a 60, BATCH_SIZE=4, y procesa los '
                  'audios >90 min por separado.')
        elif str(_ult.get('motivo', '')).startswith('fin:'):
            print('   ⇒ VEREDICTO: cierre ordenado; la sesión terminó por decisión propia '
                  f"({_ult.get('motivo')}).")
        elif _ult.get('motivo') == 'periodico':
            print('   ⇒ VEREDICTO: la telemetría se cortó en seco con RAM holgada ⇒ el corte')
            print('      fue EXTERNO (límite/cuota de Colab, desconexión del navegador).')
            print('      Acción: mantén la pestaña abierta y activa; revisa tu plan de Colab.')
        else:
            print('   ⇒ VEREDICTO: cierre ordenado o guardia de RAM; revisa el motivo arriba.')

# ── 3 · ¿Qué está frenando el rendimiento? ────────────────────────────
_ev = _leer_jsonl_a(_EV_A)
if _ev:
    _tipos = _Counter(e.get('event') for e in _ev)
    print('\n📊 Eventos acumulados en el histórico:')
    for _nombre, _n in _tipos.most_common(14):
        print(f'   {_nombre:<28} {_n:>6}')
    _motivos = _Counter()
    for _e in _ev:
        if _e.get('event') in ('quality_rejected', 'failed_retryable'):
            _motivos[str(_e.get('error'))[:90]] += 1
    if _motivos:
        print('\n🔍 Causas de rechazo/fallo más frecuentes:')
        for _causa, _n in _motivos.most_common(8):
            print(f'   {_n:>4}× {_causa}')
    _recs = [e for e in _ev if e.get('event') == 'models_recycled']
    if _recs:
        print(f'\n🔄 Reciclajes de modelo: {len(_recs)}')
        for _e in _recs[-5:]:
            print(f"   tras {_e.get('files_done')} archivos ({_e.get('reason')}): "
                  f"RAM {_e.get('ram_pct_before')}% → {_e.get('ram_pct_after')}%")
        _g = [float(_e['ram_pct_before']) - float(_e['ram_pct_after']) for _e in _recs
              if isinstance(_e.get('ram_pct_before'), (int, float))
              and isinstance(_e.get('ram_pct_after'), (int, float))]
        if _g:
            _m = sum(_g) / len(_g)
            print(f'   Recuperación media: {_m:+.1f} puntos porcentuales')
            print('   ⇒ ' + ('el reciclaje SÍ purga la fuga: sube RAM_PCT_RECICLAR si molesta.'
                             if _m >= 5 else
                             'el reciclaje NO recupera casi nada ⇒ la fuga está fuera de los '
                             'modelos. Usa MAX_ARCHIVOS_POR_SESION y reinicia por tandas.'))
    _rtfs = sorted(float(e['real_time_factor']) for e in _ev
                   if e.get('event') == 'job_metrics'
                   and isinstance(e.get('real_time_factor'), (int, float)))
    if _rtfs:
        print(f'\n⚡ RTF observado: n={len(_rtfs)} · p10={_rtfs[len(_rtfs) // 10]:.1f}x · '
              f'mediana={_rtfs[len(_rtfs) // 2]:.1f}x · máx={_rtfs[-1]:.1f}x')
        print(f'   A la mediana, 1 hora de audio cuesta ~{60 / _rtfs[len(_rtfs) // 2]:.1f} '
              'min de ASR.')
print('\n' + '═' * 70)

## 🟢 12 · Apagado manual (opcional)

El apagado **automático** ya vive en la CELDA 8 (`finally`), según tu configuración. Esta celda es para liberar la máquina a mano después de usar las herramientas (renombrar, autopsia).

In [ ]:
# 🟢 CELDA 12 — apagado MANUAL (el automático vive en la CELDA 8)
APAGAR_AHORA = False  # ⇦ True + ejecutar esta celda = libera la máquina ya

if APAGAR_AHORA:
    if not IN_COLAB:
        print('ℹ️ No es Colab: nada que apagar.')
    else:
        import time as _t3
        print(f'🛑 Apagado manual en {RETRASO_APAGADO_S} s — ⏹ cancela.')
        try:
            _r3 = int(RETRASO_APAGADO_S)
            while _r3 > 0:
                print(f'   … {_r3:>3d} s', flush=True)
                _t3.sleep(min(5, _r3))
                _r3 -= 5
        except KeyboardInterrupt:
            print('✋ Cancelado.')
        else:
            import contextlib as _ctx3
            with _ctx3.suppress(Exception):
                from google.colab import drive as _gd3
                _gd3.flush_and_unmount()
            from google.colab import runtime as _gr3
            _gr3.unassign()
else:
    print('ℹ️ Apagado manual desactivado (APAGAR_AHORA=False). '
          'El automático ya se gestionó en la CELDA 8 según tu configuración.')

---
## 📖 Referencia operativa

| Escenario | Comportamiento |
|---|---|
| Primera ejecución | Verifica e inicia en bloques de `TAMANO_BLOQUE_PLANIFICACION`: el primer `.txt` llega en minutos, no al final de hashear todo |
| Sesión cortada por Colab | Nada confirmado se pierde; el audio en curso se reinicia. Al reejecutar, `recover` rescata commits preparados y sigue |
| Archivo nuevo / modificado | Entra solo. Identidad SHA-256 **completa** calculada incrementalmente (justo antes de procesar) |
| Duplicado (misma grabación en 2 rutas) | Se copia el JSON maestro y se re-renderiza **sin GPU** (`reused`) — solo con identidad fuerte (`full`) |
| Sustitución con igual tamaño y fecha | La caza la **auditoría rotatoria** (`AUDITORIAS_ESTRICTAS_POR_SESION` × `MAX_GB_AUDITORIA_POR_SESION`, cursor durable) |
| Activas un entregable / cambias formato o encabezado | **Republicación automática en CPU** desde el JSON maestro (evento `republished`) — cero GPU |
| Cambias la plantilla de nombres | Aplica a archivos **nuevos**; los publicados conservan su nombre (estabilidad) |
| Cambias modelo / idioma / beam / hablantes / glosario | Perfil de **motor** distinto ⇒ `POLITICA_CAMBIO_PERFIL`: conservar_y_reportar · reprocesar · detener. Nunca silencio |
| Renombraste hablantes (CELDA 10) | Mapping persistente + republicación CPU; las republicaciones futuras conservan los nombres |
| Diarización falla en UN archivo | Reintento en próxima sesión (hasta `MAX_INTENTOS_POR_ARCHIVO`); la caché durable de diarización hace que el reintento solo pague ASR. En el último intento, la **válvula** publica MARCADO |
| Diarización falla GLOBALMENTE | Community-1 se valida **antes** de cargar Whisper; token/términos pre-verificados; 2 fallos consecutivos ⇒ *circuit breaker* (`abort_fatal`) — y NO apaga la VM: necesitas leer el error |
| Audio sin voz | Línea `SIN_VOZ` de primera clase, estado `ok_sin_voz`, sin gastar reintentos |
| ffprobe tropieza (Drive lento) | `retryable`: ni bloquea ni consume intentos. Solo la corrupción real marca `invalid` |
| RAM subiendo (fuga ~71 MB/archivo) | Recicla modelos al `RAM_PCT_RECICLAR` %; al `RAM_PCT_DETENER` % cierra LIMPIO. `limite_ram` nunca apaga la VM: reinicia sesión y `Ejecutar todo` |
| Dos sesiones a la vez | Lock con heartbeat; corrupto ⇒ **se niega** (no se roba) salvo `FORZAR_TOMAR_BLOQUEO`; propiedad verificada antes de cada promoción |
| Salida reutilizada con otra colección | `workspace_identity.json` liga RUTA_SALIDA a UNA entrada y aborta el cruce |
| Sesión sin pendientes | Ni GPU ni token: termina en segundos e informa |
| La sesión muere sin terminar | `telemetria.jsonl` + CELDA 11 dictaminan OOM vs corte externo |
| `FORZAR_REPROCESO=True` | Rehace todo e ignora también la caché durable de diarización |

### 📄 Estructura de salida

```
RUTA_SALIDA/
├── ClienteA/2026/reunion.txt              ⭐ entregable (espejo limpio de reunion.ts)
│                 reunion.transcript.md    (si SALIDA_TRANSCRIPT_MD)
│                 reunion.srt · reunion.json · reunion.full_for_llm.txt
│                 reunion.parte_01.txt …   (si SALIDA_SPLITS)
├── _resumen.md                            ← tabla legible, actualizada tras CADA archivo
└── .speakerscribe_state/                  ← NO borrar (reanudación e idempotencia)
    ├── events.jsonl                       ← journal transaccional
    ├── intermedios/<id>.json.gz           ← JSON maestro (republicar/renombrar sin GPU)
    ├── renombres/<id>.json                ← nombres de hablantes persistentes
    ├── pendientes_revision.json           ← marcados/bloqueados para revisar
    ├── telemetria.jsonl                   ← caja negra (CELDA 11)
    └── diar_cache/ · logs_fallos/ · audit_cursor.json · last_summary.json
```

### Formato del `.txt` (`ss-v5-render-1`)

```
# archivo_origen: ClienteA/2026/reunion.ts
# duracion: 01:32:07 · idioma: es (p=0.99)
# hablantes: 3 (SPEAKER_00, SPEAKER_01, SPEAKER_02)
[00:00:04 - 00:00:11] SPEAKER_00: Buenos días a todos, iniciamos la sesión…
[00:00:12 - 00:00:15] SPEAKER_01: Gracias por venir.
```

### Resumen de reunión con un LLM

1. Activa `SALIDA_FULL_LLM = True` (CELDA 3) — si el archivo ya está procesado, se republica **sin GPU**.
2. Abre `<nombre>.full_for_llm.txt` y pégalo con un prompt tipo: *«Con base en esta transcripción, genera un resumen estructurado: asistentes, decisiones, compromisos con responsables y fechas.»*

### Fuentes técnicas

[speakerscribe](https://github.com/EnriqueForero/speakerscribe) · [faster-whisper](https://github.com/SYSTRAN/faster-whisper) · [pyannote community-1](https://huggingface.co/pyannote/speaker-diarization-community-1) · [límites de Colab](https://research.google.com/colaboratory/faq.html#resource-limits) · [`os.replace` atómico](https://docs.python.org/3/library/os.html#os.replace)

> ⚠️ Ningún ASR/diarizador es infalible: para dominios críticos valida WER/DER con una muestra propia (`pip install 'speakerscribe[bench]'` + `speakerscribe bench`). `BATCH_SIZE=1` da la ruta secuencial bit-exacta; `8` es el perfil de cuota.